In [6]:
import os
import shutil


images_ts_dir = os.path.join('./nnUNet_raw', 'Dataset501_Vesuvius3D_Official', 'imagesTs')

test_dir = os.path.join('./nnUNet_raw', 'test_images')

for filename in os.listdir(test_dir):
    print(filename)
    if filename.endswith('.tif'):
        src = os.path.join(test_dir, filename)
        base = os.path.splitext(filename)[0]
        dst = os.path.join(images_ts_dir, f'{base}_0000.tif')
        shutil.copy(src, dst)
        print(f'Copied: {base}_0000.tif')

print('All test images copied successfully!')

1407735.tif
Copied: 1407735_0000.tif
All test images copied successfully!


In [1]:
import sys
import os
import torch
import numpy as np
import nnunetv2
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO
from batchgenerators.utilities.file_and_folder_operations import join


INPUT_FOLDER = "./nnUNet_raw/Dataset501_Vesuvius3D_Official/imagesTs"
OUTPUT_FOLDER = "./nnUNet_upload/3d_fullres_TTA"
NNUNET_RESULTS_FOLDER = "./nnUNet_results"
DATASET_NAME = "Dataset501_Vesuvius3D_Official"
TRAINER_NAME = "nnUNetTrainerMedialSurfaceRecall_MuSGD"
PLANS_NAME = "nnUNetResEncUNetMPlans"
CONFIG_NAME = "3d_192_group32_gelu"
FOLD_ID = 1

MODEL_FOLDER = join(
    NNUNET_RESULTS_FOLDER,
    DATASET_NAME,
    f"{TRAINER_NAME}__{PLANS_NAME}__{CONFIG_NAME}"
)


def inference_worker(file_list, gpu_id):
    if not file_list:
        return

    print(f"[GPU {gpu_id}] 啟動... 準備處理 {len(file_list)} 個檔案")

    # 初始化預測器
    predictor = nnUNetPredictor(
        tile_step_size=0.4,
        use_gaussian=True,
        use_mirroring=True,
        perform_everything_on_device=True,
        device=torch.device('cuda', gpu_id),
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True
    )

    predictor.initialize_from_trained_model_folder(
        MODEL_FOLDER,
        use_folds=(FOLD_ID,),
        checkpoint_name='checkpoint_best.pth',
    )

    rw = SimpleITKIO()

    for filename in file_list:
        full_input_path = join(INPUT_FOLDER, filename)

        # 檔名處理: 去除 _0000
        name, ext = os.path.splitext(filename)
        clean_name = name.replace("_0000", "")
        output_filename = clean_name + ext
        full_output_path = join(OUTPUT_FOLDER, output_filename)

        print(f"[GPU {gpu_id}] 正在推理: {clean_name}")

        try:
            # 修改處：save_probabilities 設為 False，直接獲取分割結果
            result = predictor.predict_from_files(
                list_of_lists_or_source_folder=[[full_input_path]],
                output_folder_or_list_of_truncated_output_files=None,
                save_probabilities=False
            )

            # 修改處：直接取用預測出的 Label (通常在 index 0)
            final_seg = result[0].astype(np.uint8)

            # 確保維度正確 (如果是 2D 則補上通道)
            if final_seg.ndim == 2:
                final_seg = final_seg[None, ...]

            dummy_props = {
                'sitk_stuff': {
                    'spacing': (1.0, 1.0, 1.0),
                    'origin': (0.0, 0.0, 0.0),
                    'direction': (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
                },
                'spacing': [1.0, 1.0, 1.0]
            }

            rw.write_seg(final_seg, full_output_path, dummy_props)
            print(f"[GPU {gpu_id}] 完成儲存: {output_filename}")

        except Exception as e:
            print(f"[GPU {gpu_id}] 發生錯誤 ({filename}): {e}")

In [3]:
import os
import torch
from batchgenerators.utilities.file_and_folder_operations import subfiles, maybe_mkdir_p

# ==========================================
# 3. 執行管理
# ==========================================
def run_inference_manager():
    if not os.path.exists(MODEL_FOLDER):
        raise FileNotFoundError(f"Model folder not found: {MODEL_FOLDER}")

    # 掃描輸入資料夾
    if not os.path.exists(INPUT_FOLDER):
        print(f"Warning: Input folder does not exist: {INPUT_FOLDER}")
        return

    files = subfiles(INPUT_FOLDER, suffix='.tif', join=False)
    files.sort()
    maybe_mkdir_p(OUTPUT_FOLDER)

    num_files = len(files)
    print(f"=== Job Started ===")
    print(f"Files found: {num_files}")

    if num_files == 0:
        print("No files to process.")
        return

    # === 核心執行邏輯：無論檔案數量，固定使用單卡 (GPU 0) 循序處理 ===
    print(f"Mode: Sequential -> Using single GPU (GPU 0) for all {num_files} files.")

    # 直接呼叫推論函式，不產生新的 Process
    inference_worker(files, 0)

    print("All GPU jobs completed.")

if __name__ == "__main__":
    run_inference_manager()

=== Job Started ===
Files found: 1
Mode: Sequential -> Using single GPU (GPU 0) for all 1 files.
[GPU 0] 啟動... 準備處理 1 個檔案
[GPU 0] 正在推理: 1407735
There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict
WARNING no spacing file found for images ['./nnUNet_raw/Dataset501_Vesuvius3D_Official/imagesTs/1407735_0000.tif']
Assuming spacing (1, 1, 1).

Predicting image of shape torch.Size([1, 320, 314, 314]):
perform_everything_on_device: True


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 27/27 [00:23<00:00,  1.13it/s]


sending off prediction to background worker for resampling

Done with image of shape torch.Size([1, 320, 314, 314]):
[GPU 0] 完成儲存: 1407735.tif
All GPU jobs completed.


In [5]:
import math
import multiprocessing
from itertools import product
from typing import List, Tuple

import cv2
from numba import njit
from skimage.morphology import skeletonize, remove_small_objects
from skimage.measure import label
from skimage.draw import line as skimage_line
from scipy.ndimage import convolve, distance_transform_edt, binary_closing, binary_fill_holes, binary_dilation, \
    gaussian_filter
import numpy as np
from scipy import ndimage as ndi, ndimage
from scipy.ndimage import find_objects, binary_fill_holes
from skimage.morphology import remove_small_objects, ball
from skimage.draw import line
from skimage.measure import label as measure_label
from skimage.segmentation import watershed, find_boundaries
from collections import Counter

# from topometrics._bm_loader import load_betti_matching

# 嘗試導入路徑計算模組
try:
    from skimage.graph import route_through_array
except ImportError:
    from skimage.graph import MCP_Geometric


    def route_through_array(cost, start, end):
        mcp = MCP_Geometric(cost)
        cumulative_costs, traceback = mcp.find_costs([start], [end])
        return mcp.traceback(end), 0


# =========================================================================
#  Numba 加速核心 (Geodesic, PCA, Ray-Casting)
# =========================================================================

@njit(cache=True)
def compute_geodesic_distance_numba(mask, start_point):
    """ Geodesic Distance Transform"""
    d, h, w = mask.shape
    dist_map = np.full((d, h, w), np.inf, dtype=np.float32)
    sz, sy, sx = start_point

    if mask[sz, sy, sx] == 0:
        found = False
        for dz in range(-1, 2):
            for dy in range(-1, 2):
                for dx in range(-1, 2):
                    nz, ny, nx = sz + dz, sy + dy, sx + dx
                    if 0 <= nz < d and 0 <= ny < h and 0 <= nx < w:
                        if mask[nz, ny, nx] != 0:
                            sz, sy, sx = nz, ny, nx
                            found = True
                            break
                if found: break
            if found: break
        if not found: return np.zeros((d, h, w), dtype=np.float32)

    max_queue_len = d * h * w
    queue_z = np.empty(max_queue_len, dtype=np.int32)
    queue_y = np.empty(max_queue_len, dtype=np.int32)
    queue_x = np.empty(max_queue_len, dtype=np.int32)
    head, tail = 0, 0

    queue_z[tail], queue_y[tail], queue_x[tail] = sz, sy, sx
    tail += 1
    dist_map[sz, sy, sx] = 0.0

    dz_off = np.array([0, 0, 0, 0, 1, -1], dtype=np.int8)
    dy_off = np.array([0, 0, 1, -1, 0, 0], dtype=np.int8)
    dx_off = np.array([1, -1, 0, 0, 0, 0], dtype=np.int8)

    while head < tail:
        cz, cy, cx = queue_z[head], queue_y[head], queue_x[head]
        head += 1
        next_dist = dist_map[cz, cy, cx] + 1.0

        for i in range(6):
            nz, ny, nx = cz + int(dz_off[i]), cy + int(dy_off[i]), cx + int(dx_off[i])
            if 0 <= nz < d and 0 <= ny < h and 0 <= nx < w:
                if mask[nz, ny, nx] != 0 and dist_map[nz, ny, nx] == np.inf:
                    dist_map[nz, ny, nx] = next_dist
                    queue_z[tail], queue_y[tail], queue_x[tail] = nz, ny, nx
                    tail += 1
    return dist_map


@njit(fastmath=True)
def compute_pca_field_numba(labels, radius, step=1):
    h, w = labels.shape
    norm_y = np.zeros((h, w), dtype=np.float32)
    norm_x = np.zeros((h, w), dtype=np.float32)

    for r in range(0, h, step):
        for c in range(0, w, step):
            current_id = labels[r, c]
            if current_id == 0: continue
            sum_xx, sum_yy, sum_xy, count = 0.0, 0.0, 0.0, 0.0
            r_min, r_max = max(0, r - radius), min(h, r + radius + 1)
            c_min, c_max = max(0, c - radius), min(w, c + radius + 1)
            for nr in range(r_min, r_max):
                for nc in range(c_min, c_max):
                    if labels[nr, nc] == current_id:
                        dy, dx = nr - r, nc - c
                        sum_xx += dx * dx
                        sum_yy += dy * dy
                        sum_xy += dx * dy
                        count += 1.0
            if count < 3: continue
            A, B, C = sum_yy, sum_xy, sum_xx
            tangent_angle = 0.5 * np.arctan2(2 * B, C - A)
            normal_angle = tangent_angle + np.pi / 2.0
            norm_y[r, c] = np.sin(normal_angle)
            norm_x[r, c] = np.cos(normal_angle)
    return norm_y, norm_x


@njit(fastmath=True)
def find_first_collision_pair(y_coords, x_coords, pca_ny, pca_nx, mask, labels, ray_len):
    h, w = mask.shape
    num_points = len(y_coords)
    for i in range(num_points):
        r, c = y_coords[i], x_coords[i]
        ny, nx = pca_ny[r, c], pca_nx[r, c]
        if ny == 0.0 and nx == 0.0: continue
        source_id = labels[r, c]
        for sign in (1, -1):
            has_left_self = False
            for t in range(1, ray_len):
                cy = int(r + sign * t * ny + 0.5)
                cx = int(c + sign * t * nx + 0.5)
                if cy < 0 or cy >= h or cx < 0 or cx >= w: break
                pixel_val = mask[cy, cx]
                target_id = labels[cy, cx]
                if not has_left_self:
                    if pixel_val == 0:
                        has_left_self = True
                    elif target_id != source_id:
                        return True, r, c, cy, cx
                else:
                    if pixel_val > 0:
                        if target_id != source_id:
                            return True, r, c, cy, cx
                        else:
                            break
    return False, 0, 0, 0, 0


@njit(fastmath=True)
def find_all_collision_pairs(y_coords, x_coords, pca_ny, pca_nx, mask, labels, ray_len):
    """
    修改版：不再找到第一個就停，而是收集所有碰撞對。
    回傳: results list, 每個元素為 (r1, c1, r2, c2)
    """
    h, w = mask.shape
    num_points = len(y_coords)

    # 建立一個 List 來儲存結果
    # Numba 會自動推斷這是 List(Tuple(int64, int64, int64, int64))
    results = []

    for i in range(num_points):
        r, c = y_coords[i], x_coords[i]
        ny, nx = pca_ny[r, c], pca_nx[r, c]

        # 如果 PCA 向量為 0 (無法計算方向)，跳過
        if ny == 0.0 and nx == 0.0: continue

        source_id = labels[r, c]

        # 標記這個點是否已經找到對象，避免同一個點向左向右都加，造成重複 (視需求可保留雙向)
        # 這裡設定為：如果正向找到就不找反向，確保每個起始點最多貢獻一條路徑
        found_for_this_point = False

        for sign in (1, -1):
            has_left_self = False
            for t in range(1, ray_len):
                cy = int(r + sign * t * ny + 0.5)
                cx = int(c + sign * t * nx + 0.5)

                # 邊界檢查
                if cy < 0 or cy >= h or cx < 0 or cx >= w: break

                pixel_val = mask[cy, cx]
                target_id = labels[cy, cx]

                if not has_left_self:
                    # 還沒離開自己
                    if pixel_val == 0:
                        has_left_self = True
                    elif target_id != source_id:
                        # 緊鄰就是不同 ID (沾黏嚴重)，視為碰撞
                        results.append((r, c, cy, cx))
                        found_for_this_point = True
                        break
                else:
                    # 已經離開自己 (在背景中移動)
                    if pixel_val > 0:
                        # 撞到某個東西
                        if target_id != source_id:
                            # 撞到別人 -> 有效切割對
                            results.append((r, c, cy, cx))
                            found_for_this_point = True
                            break
                        else:
                            # 撞回自己 (U型彎曲) -> 無效，停止這條射線
                            break

            if found_for_this_point:
                break

    return results


# =========================================================================
#  Python 輔助 (Seeds, Path)
# =========================================================================


def get_split_seeds(mask_2d, ray_len=64, pca_radius=8, pca_step=3):
    mask_uint8 = (mask_2d > 0).astype(np.uint8)
    y_coords, x_coords = np.where(mask_uint8 > 0)

    if len(y_coords) < 10: return []  # 改回傳空 list

    # 降採樣取點 (Step sampling)
    valid_indices = (y_coords % pca_step == 0) & (x_coords % pca_step == 0)
    y_sampled = y_coords[valid_indices]
    x_sampled = x_coords[valid_indices]

    if len(y_sampled) == 0: return []

    _, labels_map = cv2.connectedComponents(mask_uint8, connectivity=8)
    labels_map = labels_map.astype(np.int32)

    # 計算 PCA 場
    pca_ny, pca_nx = compute_pca_field_numba(labels_map, pca_radius, step=pca_step)

    # 使用新的 Numba 函數取得所有碰撞對
    raw_results = find_all_collision_pairs(
        y_sampled, x_sampled, pca_ny, pca_nx, mask_uint8, labels_map, ray_len
    )

    # 格式化輸出 [((y1,x1), (y2,x2)), ...]
    seeds_list = []
    for (r1, c1, r2, c2) in raw_results:
        seeds_list.append(((r1, c1), (r2, c2)))

    return seeds_list


#
# def analyze_shortest_path(mask_3d, start_pt, end_pt):
#     """
#     優化版最短路徑分析：
#     1. 僅對 Z, Y 軸進行 2 倍降採樣，X 軸保持原解析度 (Scale=1)。
#     2. 加入 K-Neighbor Search 容錯。
#     3. 回傳還原後的 Full-Resolution Path 供視覺化。
#     """
#     # 1. 如果 Mask 很小，直接算原圖
#     if mask_3d.shape[0] < 20 or mask_3d.shape[1] < 20:
#         return _analyze_path_original(mask_3d, start_pt, end_pt)
#
#     # ==========================================
#     # 設定降採樣步長： (Z=2, Y=2, X=1)
#     # ==========================================
#     step_z, step_y, step_x = 2, 2, 1
#
#     small_mask = mask_3d[::step_z, ::step_y, ::step_x]
#
#     # 2. 映射 Start/End 座標到小圖空間
#     d_s, h_s, w_s = small_mask.shape
#
#     def to_small(pt):
#         sz = min(pt[0] // step_z, d_s - 1)
#         sy = min(pt[1] // step_y, h_s - 1)
#         sx = min(pt[2] // step_x, w_s - 1)
#         return (sz, sy, sx)
#
#     small_start = to_small(start_pt)
#     small_end = to_small(end_pt)
#
#     # 3. 在小圖上計算距離變換 (EDT) 與路徑
#     dist = distance_transform_edt(small_mask)
#     cost_map = np.max(dist) - dist + 1
#     cost_map[small_mask == 0] = np.inf
#
#     try:
#         indices, _ = route_through_array(cost_map, small_start, small_end)
#         path_indices = np.array(indices)
#
#         path_len = len(path_indices)
#         if path_len == 0:
#             return _analyze_path_original(mask_3d, start_pt, end_pt)
#
#         # =======================================================
#         # K-Neighbor Search 尋找有效 Midpoint
#         # =======================================================
#         mid_idx = path_len // 2
#         search_k = 5
#         valid_midpoint = None
#
#         # 產生搜尋順序: 0, 1, -1, 2, -2 ...
#         offsets = [0]
#         for i in range(1, search_k + 1):
#             offsets.append(i)
#             offsets.append(-i)
#
#         for offset in offsets:
#             current_idx = mid_idx + offset
#             if 0 <= current_idx < path_len:
#                 small_pt = path_indices[current_idx]
#
#                 # 映射回原始尺寸 (針對單點)
#                 real_z = small_pt[0] * step_z + (step_z // 2)
#                 real_y = small_pt[1] * step_y + (step_y // 2)
#                 real_x = small_pt[2] * step_x
#
#                 # 邊界檢查
#                 real_z = np.clip(real_z, 0, mask_3d.shape[0] - 1)
#                 real_y = np.clip(real_y, 0, mask_3d.shape[1] - 1)
#                 real_x = np.clip(real_x, 0, mask_3d.shape[2] - 1)
#
#                 candidate_pt = (int(real_z), int(real_y), int(real_x))
#
#                 if mask_3d[candidate_pt] > 0:
#                     valid_midpoint = candidate_pt
#                     break
#
#         if valid_midpoint is not None:
#             # ===================================================
#             # 新增：還原整條路徑座標 (Upscaling Path)
#             # ===================================================
#             full_res_path = path_indices.copy()
#
#             # 向量化計算：還原座標並加上中心偏移量
#             # Z 軸
#             full_res_path[:, 0] = full_res_path[:, 0] * step_z + (step_z // 2)
#             # Y 軸
#             full_res_path[:, 1] = full_res_path[:, 1] * step_y + (step_y // 2)
#             # X 軸 (不變)
#             full_res_path[:, 2] = full_res_path[:, 2] * step_x
#
#             # 統一進行邊界限制 (Clip)，防止偏移後超出原圖範圍
#             max_coords = np.array(mask_3d.shape) - 1
#             # 利用 numpy 的廣播機制將所有點限制在 [0, max]
#             # 注意：np.clip 需要 array 為 float 或 int，這裡保持 int
#             full_res_path[:, 0] = np.clip(full_res_path[:, 0], 0, max_coords[0])
#             full_res_path[:, 1] = np.clip(full_res_path[:, 1], 0, max_coords[1])
#             full_res_path[:, 2] = np.clip(full_res_path[:, 2], 0, max_coords[2])
#
#             return full_res_path, valid_midpoint
#         else:
#             return _analyze_path_original(mask_3d, start_pt, end_pt)
#
#     except Exception as e:
#         print(e)
#         print(f"Analyze Path Error: {start_pt} -> {end_pt}")
#         return _analyze_path_original(mask_3d, start_pt, end_pt)


def _analyze_path_original(mask_3d, start_pt, end_pt):
    """ 原始的全解析度路徑分析 (Fallback) """
    dist = distance_transform_edt(mask_3d)
    cost_map = np.max(dist) - dist + 1
    cost_map[mask_3d == 0] = np.inf
    try:
        indices, _ = route_through_array(cost_map, start_pt, end_pt)
        path_indices = np.array(indices)
        if len(path_indices) == 0:
            print(f"找不到路徑: {start_pt} -> {end_pt}")
            return None, None
        midpoint = tuple(map(int, path_indices[len(path_indices) // 2]))
        return path_indices, midpoint
    except Exception as e:
        print(f"Analyze Path Error: {start_pt} -> {end_pt}")
        print(e)
        return None, None


# =========================================================================
#  主函數: split_paper (使用 High Conf Region 作為 Watershed Markers)
# =========================================================================

def get_split_paper(mask_3d, high_conf_labels, ray_len=64, max_iter=5, cleanup_iter=2):
    """
    對 3D Binary Mask 執行迭代式幾何分割。

    1. 遍歷 Slice 找尋候選切割點。
    2. 投票找出最主要的 High Conf 連通域組合 (Pair A-B)。
    3. 計算分割幾何邊界：透過 0, Mid, End 的種子路徑計算中點，建立 Geodesic Basin。
    4. 【核心改動】建立 Markers：直接使用 Pair A 和 Pair B 在此 ROI 內的像素作為種子區域。
    5. 執行 Watershed。
    """

    if mask_3d.dtype == bool or np.max(mask_3d) == 1:
        refined_labels = measure_label(mask_3d)
    else:
        refined_labels = measure_label(mask_3d > 0)

    refined_labels = refined_labels.astype(np.int32)
    next_new_label = refined_labels.max() + 1
    debug_paths = []

    print(f">> 啟動 split_paper: Ray={ray_len}, Iter={max_iter}, Cleanup={cleanup_iter}")

    for iteration in range(max_iter):
        unique_ids = np.unique(refined_labels)
        unique_ids = unique_ids[unique_ids != 0]
        splits_this_round = 0

        for obj_id in unique_ids:
            obj_mask = (refined_labels == obj_id)
            slices = np.where(obj_mask)
            if len(slices[0]) == 0: continue

            # 取得目前的 Bounding Box
            z_min, z_max = np.min(slices[0]), np.max(slices[0])
            y_min, y_max = np.min(slices[1]), np.max(slices[1])
            x_min, x_max = np.min(slices[2]), np.max(slices[2])

            roi_mask = obj_mask[z_min:z_max + 1, y_min:y_max + 1, x_min:x_max + 1]
            d, h, w = roi_mask.shape

            # -----------------------------------------------------------
            # 1. 遍歷所有 Slice 進行 Dense Sampling 並做投票
            # -----------------------------------------------------------
            candidate_seeds = []

            for z_local in range(d):
                slice_2d = roi_mask[z_local, :, :]

                # 這裡現在會回傳一個 List，包含該層所有的切割候選
                seeds_list = get_split_seeds(slice_2d, ray_len=ray_len)

                # 如果這層有找到種子，遍歷它們
                if seeds_list:
                    for seeds in seeds_list:
                        (y1, x1), (y2, x2) = seeds

                        # 轉全域座標查表
                        z_global = z_min + z_local
                        y1_global, x1_global = y_min + y1, x_min + x1
                        y2_global, x2_global = y_min + y2, x_min + x2

                        # 邊界檢查
                        if not (0 <= y1_global < high_conf_labels.shape[1] and 0 <= x1_global < high_conf_labels.shape[
                            2]): continue
                        if not (0 <= y2_global < high_conf_labels.shape[1] and 0 <= x2_global < high_conf_labels.shape[
                            2]): continue

                        id_a = high_conf_labels[z_global, y1_global, x1_global]
                        id_b = high_conf_labels[z_global, y2_global, x2_global]

                        # 核心邏輯：必須連接兩個不同的高信心物件
                        if id_a != 0 and id_b != 0 and id_a != id_b:
                            pair = tuple(sorted((id_a, id_b)))
                            candidate_seeds.append({
                                'z_local': z_local,
                                'start_local': (z_local, y1, x1),
                                'end_local': (z_local, y2, x2),
                                'conf_pair': pair
                            })

            # -----------------------------------------------------------
            # 2. 投票選出最佳組合 (A-B)
            # -----------------------------------------------------------
            if not candidate_seeds:
                continue

            pair_counts = Counter([c['conf_pair'] for c in candidate_seeds])
            best_pair, count = pair_counts.most_common(1)[0]
            id_comp_1, id_comp_2 = best_pair  # 取出這兩個高信心度連通域的 ID

            # 僅保留最佳組合的種子
            filtered_seeds = [c for c in candidate_seeds if c['conf_pair'] == best_pair]

            # -----------------------------------------------------------
            # 3. 排序 (按照 y, z, x) 並選取 3 個 (計算 Basin 幾何中心用)
            # -----------------------------------------------------------
            filtered_seeds.sort(key=lambda x: (x['start_local'][1], x['start_local'][0], x['start_local'][2]))

            num_seeds = len(filtered_seeds)
            if num_seeds == 0:
                continue

            indices_to_pick = sorted(list(set([0, num_seeds // 2, num_seeds - 1])))

            valid_midpoints = []

            for idx in indices_to_pick:
                seed_info = filtered_seeds[idx]
                p1 = seed_info['start_local']
                p2 = seed_info['end_local']
                # 計算路徑求 Midpoint (定義邊界位置)
                path_idx, midpoint = _analyze_path_original(roi_mask, p1, p2)

                if path_idx is not None:
                    valid_midpoints.append(midpoint)

                    # 存入 Debug Path 供顯示
                    global_path = path_idx + np.array([z_min, y_min, x_min])
                    debug_paths.append(global_path)
                else:
                    print("path_idx is None")

            if not valid_midpoints:
                continue

            # -----------------------------------------------------------
            # 4. 計算 Geodesic Distance Field (基於 Midpoints)
            # -----------------------------------------------------------
            # 這定義了 "哪裡是分界線" (脊線 = 0)
            fused_dist_map = np.full(roi_mask.shape, np.inf, dtype=np.float32)
            roi_mask_int = roi_mask.astype(np.uint8)

            for midpoint in valid_midpoints:
                start_pt = (int(midpoint[0]), int(midpoint[1]), int(midpoint[2]))
                d_map = compute_geodesic_distance_numba(roi_mask_int, start_pt)
                fused_dist_map = np.minimum(fused_dist_map, d_map)

            fused_dist_map[fused_dist_map == np.inf] = 0

            # -----------------------------------------------------------
            # 5. 【修改】使用 High Confidence Regions 作為 Markers
            # -----------------------------------------------------------
            basin_map = -1.0 * fused_dist_map

            # 取出目前 ROI 範圍內的 High Conf Labels
            high_conf_roi = high_conf_labels[z_min:z_max + 1, y_min:y_max + 1, x_min:x_max + 1]

            markers = np.zeros_like(roi_mask, dtype=np.int32)

            # 將 High Conf ID A 設為 Label 1
            markers[(high_conf_roi == id_comp_1) & (roi_mask > 0)] = 1

            # 將 High Conf ID B 設為 Label 2
            markers[(high_conf_roi == id_comp_2) & (roi_mask > 0)] = 2

            # 確保有兩個標記才執行
            unique_markers = np.unique(markers)
            if 1 in unique_markers and 2 in unique_markers:

                # 執行分水嶺
                labels_ws = watershed(basin_map, markers, mask=roi_mask)

                if np.max(labels_ws) >= 2:
                    # 選定要分離的部分 (這裡是 Label 2)
                    split_mask_2 = (labels_ws == 2)

                    roi_view = refined_labels[z_min:z_max + 1, y_min:y_max + 1, x_min:x_max + 1]
                    roi_view[split_mask_2] = next_new_label

                    # 邊界清理
                    mask_old = (roi_view == obj_id)
                    mask_new = (roi_view == next_new_label)
                    struct_26 = np.ones((3, 3, 3), dtype=bool)

                    neighbors_have_old = binary_dilation(mask_old, structure=struct_26, iterations=cleanup_iter)
                    boundary_pixels_in_new = mask_new & neighbors_have_old

                    neighbors_have_new = binary_dilation(mask_new, structure=struct_26, iterations=cleanup_iter)
                    boundary_pixels_in_old = mask_old & neighbors_have_new

                    roi_view[boundary_pixels_in_new] = 0
                    roi_view[boundary_pixels_in_old] = 0

                    next_new_label += 1
                    splits_this_round += 1

        print(f"   Iter {iteration + 1}/{max_iter}: 分割了 {splits_this_round} 處。")
        if splits_this_round == 0:
            break

    final_binary = (refined_labels > 0).astype(np.uint8)
    return final_binary, debug_paths


def get_split_paper_8slice(vol, high_conf_labels, ray_len=64, max_iter=5, cleanup_iter=2):
    """
    將 3D 體積切分為 8 個區塊分別執行 get_split_paper。
    """
    output_vol = np.zeros_like(vol, dtype=np.uint8)
    all_debug_paths = []

    # 取得 8 個區塊的 slice (2, 2, 2)
    slices = _octant_slices(vol.shape, (2, 2, 2))

    for slc in slices:
        vol_block = vol[slc]
        # 如果區塊內沒有 mask，直接跳過
        if not np.any(vol_block):
            continue

        labels_block = high_conf_labels[slc]

        # 執行原本的分割邏輯
        refined_block, block_paths = get_split_paper(
            vol_block,
            labels_block,
            ray_len=ray_len,
            max_iter=max_iter,
            cleanup_iter=cleanup_iter
        )

        # 將結果填回
        output_vol[slc] = refined_block

        # 校正 Debug 路徑座標 (加上 slice 的起始偏移量)
        offset = np.array([slc[0].start, slc[1].start, slc[2].start])
        for path in block_paths:
            all_debug_paths.append(path + offset)

    return output_vol, all_debug_paths


def get_outward_vector(skel, ep_xy, pca_vec):
    """計算骨架端點的朝外向量"""
    start_x, start_y = int(ep_xy[0]), int(ep_xy[1])
    h, w = skel.shape
    STEPS = 5
    visited = set()
    visited.add((start_y, start_x))
    curr_x, curr_y = start_x, start_y
    for _ in range(STEPS):
        found_next = False
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                if dx == 0 and dy == 0: continue
                ny, nx = curr_y + dy, curr_x + dx
                if ny < 0 or ny >= h or nx < 0 or nx >= w: continue
                if skel[ny, nx] > 0 and (ny, nx) not in visited:
                    curr_x, curr_y = nx, ny
                    visited.add((ny, nx))
                    found_next = True
                    break
            if found_next: break
        if not found_next: break
    ref_vec_x, ref_vec_y = start_x - curr_x, start_y - curr_y
    if ref_vec_x == 0 and ref_vec_y == 0: return pca_vec
    if ref_vec_x * pca_vec[0] + ref_vec_y * pca_vec[1] < 0: return -pca_vec
    return pca_vec


def check_path_blocked(p1, p2, mask_img):
    """檢查兩點之間的路徑是否被障礙物阻擋"""
    x1, y1 = int(p1[0]), int(p1[1])
    x2, y2 = int(p2[0]), int(p2[1])
    rr, cc = skimage_line(y1, x1, y2, x2)
    valid_mask = (rr >= 0) & (rr < mask_img.shape[0]) & (cc >= 0) & (cc < mask_img.shape[1])
    rr, cc = rr[valid_mask], cc[valid_mask]
    if len(rr) == 0: return False
    safe_radius_sq = 4 ** 2

    def is_obstacle(px, py):
        if px < 0 or px >= mask_img.shape[1] or py < 0 or py >= mask_img.shape[0]: return False
        if mask_img[py, px] == 0: return False
        d1 = (px - x1) ** 2 + (py - y1) ** 2
        d2 = (px - x2) ** 2 + (py - y2) ** 2
        return (d1 > safe_radius_sq) and (d2 > safe_radius_sq)

    for i in range(len(rr)):
        curr_y, curr_x = rr[i], cc[i]
        if is_obstacle(curr_x, curr_y): return True
        if i > 0:
            prev_y, prev_x = rr[i - 1], cc[i - 1]
            if abs(curr_x - prev_x) == 1 and abs(curr_y - prev_y) == 1:
                if is_obstacle(prev_x, curr_y) and is_obstacle(curr_x, prev_y): return True
    return False


def get_pca_tangent(skeleton, center_yx, radius=16, line_len=16):
    """使用 PCA 計算端點的切線方向"""
    #
    cy, cx = center_yx
    h, w = skeleton.shape
    y_min, y_max = max(0, cy - radius), min(h, cy + radius + 1)
    x_min, x_max = max(0, cx - radius), min(w, cx + radius + 1)
    roi = skeleton[y_min:y_max, x_min:x_max].copy()
    if not np.any(roi): return None
    roi_labels = label(roi > 0, connectivity=2)
    local_cy, local_cx = cy - y_min, cx - x_min
    target_label = roi_labels[local_cy, local_cx]
    if target_label == 0: return None
    pts_y, pts_x = np.where(roi_labels == target_label)
    pts_global_y, pts_global_x = pts_y + y_min, pts_x + x_min
    dist_sq = (pts_global_y - cy) ** 2 + (pts_global_x - cx) ** 2
    mask = dist_sq <= (radius ** 2)
    valid_y, valid_x = pts_global_y[mask], pts_global_x[mask]
    if len(valid_x) < 2: return None
    data = np.vstack([valid_x, valid_y]).T
    mean = np.mean(data, axis=0)
    cov = np.cov((data - mean).T)
    if np.isnan(cov).any() or np.isinf(cov).any(): return None
    eig_vals, eig_vecs = np.linalg.eigh(cov)
    principal_vec = eig_vecs[:, -1]
    vec_len = np.linalg.norm(principal_vec)
    if vec_len == 0: return None
    vx, vy = principal_vec / vec_len
    pt1 = (int(cx - vx * line_len), int(cy - vy * line_len))
    pt2 = (int(cx + vx * line_len), int(cy + vy * line_len))
    return pt1, pt2


def get_endpoints(mask_2d):
    """取得 2D mask 的骨架端點"""
    #
    if not np.any(mask_2d): return [], None
    skeleton = skeletonize(mask_2d.astype(bool))
    skeleton_uint8 = skeleton.astype(np.uint8) * 255
    kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=np.uint8)
    neighbors = convolve(skeleton.astype(np.uint8), kernel, mode='constant', cval=0)
    coords = np.argwhere(skeleton & (neighbors <= 1))
    if len(coords) == 0: return [], skeleton_uint8
    raw_eps = [[pt[1], pt[0]] for pt in coords]
    raw_eps.sort(key=lambda p: (p[1], p[0]))
    return raw_eps, skeleton_uint8


def compute_candidate_info(all_eps, skel_img, radius=16):
    """計算所有候選端點的詳細資訊 (位置、向量)"""
    ep_info_list = []
    for i, (x, y) in enumerate(all_eps):
        t_pts = get_pca_tangent(skel_img, (y, x), radius=radius, line_len=radius)
        if not t_pts: continue
        pt1, pt2 = t_pts
        vx, vy = pt2[0] - pt1[0], pt2[1] - pt1[1]
        norm = np.sqrt(vx ** 2 + vy ** 2)
        if norm == 0: continue
        pca_vec = np.array([vx / norm, vy / norm])
        out_vec = get_outward_vector(skel_img, (x, y), pca_vec)
        ep_info_list.append({'id': i, 'xy': np.array([x, y]), 'vec': out_vec, 'tangent_pts': t_pts})
    return ep_info_list


def dynamic_filter(ep_info_list, skel_img, use_angle=True, use_block=True, use_pair=True, max_dist=15):
    """
    篩選並配對需要連接的端點

    Args:
        max_dist (float): 允許連線的最大距離 (預設 15)
    """
    num_eps = len(ep_info_list)
    if num_eps < 2: return []

    candidate_links = []
    ANGLE_WEIGHT = 10.0
    COS_THRES = np.cos(np.deg2rad(60))

    for i in range(num_eps):
        for j in range(i + 1, num_eps):
            p1 = ep_info_list[i]
            p2 = ep_info_list[j]
            vec_p1_to_p2 = p2['xy'] - p1['xy']
            dist = np.linalg.norm(vec_p1_to_p2)

            # --- [修改點 1] ---
            # 距離檢查：如果距離為 0 或大於等於 max_dist (15)，則跳過
            if dist == 0: continue
            if dist >= max_dist: continue
            # ------------------

            dir_1_to_2 = vec_p1_to_p2 / dist
            dir_2_to_1 = -dir_1_to_2
            cos_p1 = np.dot(p1['vec'], dir_1_to_2)
            cos_p2 = np.dot(p2['vec'], dir_2_to_1)

            if use_angle and (cos_p1 < COS_THRES or cos_p2 < COS_THRES): continue
            if use_block and check_path_blocked(tuple(p1['xy']), tuple(p2['xy']), skel_img): continue

            avg_cos = (cos_p1 + cos_p2) / 2.0
            score = dist * (1.0 + ANGLE_WEIGHT * (1.0 - min(avg_cos, 1.0)))
            candidate_links.append((score, i, j))

    candidate_links.sort(key=lambda x: x[0])
    matched_indices = set()
    final_connections = []

    if use_pair:
        for score, idx1, idx2 in candidate_links:
            if idx1 in matched_indices or idx2 in matched_indices: continue
            p1, p2 = ep_info_list[idx1], ep_info_list[idx2]
            # 再次檢查遮擋
            if use_block and check_path_blocked(tuple(p1['xy']), tuple(p2['xy']), skel_img): continue
            matched_indices.add(idx1)
            matched_indices.add(idx2)
            final_connections.append((p1, p2))
    else:
        pass

    return final_connections


def execute_line_repair(vol_3d, settings=None):
    """
    執行 3D 體積的逐層補線操作
    """
    # --- [修改點 2] ---
    # 設定預設值，確保 max_dist 存在
    default_settings = {
        'use_angle': True,
        'use_block': True,
        'use_pair': True,
        'trim_ends': True,
        'curve_str': 50,
        'max_dist': 8  # 預設距離限制
    }

    if settings is None:
        settings = default_settings
    else:
        # 若使用者傳入部分 settings，補齊未傳入的預設值
        for k, v in default_settings.items():
            if k not in settings:
                settings[k] = v
    # ------------------

    print(f"正在執行斷線修復 (Line Endpoint Repair), Max Dist: {settings['max_dist']}...")

    if vol_3d.dtype == bool:
        labeled_vol = label(vol_3d)
    else:
        labeled_vol = label(vol_3d > 0)

    patched_vol = (vol_3d > 0).astype(np.uint8) * 255
    d, h, w = labeled_vol.shape

    for z in range(d):
        slice_labels = labeled_vol[z, :, :]
        if not np.any(slice_labels): continue

        slice_img = patched_vol[z, :, :]
        present_ids = np.unique(slice_labels)
        present_ids = present_ids[present_ids > 0]

        for pid in present_ids:
            obj_mask = (slice_labels == pid)
            raw_eps, skel_img = get_endpoints(obj_mask)
            if len(raw_eps) < 2: continue

            ep_info_all = compute_candidate_info(raw_eps, skel_img, radius=16)

            candidates = list(ep_info_all)
            # 這裡保留原有的 trim_ends 邏輯，如果需要的話
            if settings.get('trim_ends', False) and len(candidates) > 2:
                candidates.sort(key=lambda p: (p['xy'][1], p['xy'][0]))
                candidates = candidates[1:-1]  # 去頭去尾

            # --- [修改點 3] ---
            # 將 max_dist 傳入 dynamic_filter
            connections = dynamic_filter(
                candidates, skel_img,
                use_angle=settings['use_angle'],
                use_block=settings['use_block'],
                use_pair=settings['use_pair'],
                max_dist=settings['max_dist']
            )
            # ------------------

            for (p1_obj, p2_obj) in connections:
                pt1 = tuple(map(int, p1_obj['xy']))
                pt2 = tuple(map(int, p2_obj['xy']))
                cv2.line(slice_img, pt1, pt2, 255, 1, cv2.LINE_AA)

        patched_vol[z, :, :] = slice_img

    return patched_vol


def execute_line_repair_8slice(vol_3d, settings=None, splits=(2, 2, 2), pad_size=32):
    """
    針對 execute_line_repair 的分塊並行/依序處理版本。
    自動處理 Padding 以避免在切割邊界產生錯誤的端點判定。

    Args:
        vol_3d (np.ndarray): 原始 3D 陣列 (Boolean 或 Label)。
        settings (dict): 傳遞給 execute_line_repair 的參數設定。
        splits (tuple): (z, y, x) 切分份數，預設 (2, 2, 2) 為 8 塊。
        pad_size (int): 擴充邊界大小。
                        注意：必須大於 execute_line_repair 內部的 radius (預設16)，
                        建議設為 32 或更大以確保向量計算正確。

    Returns:
        np.ndarray: 修復後的完整 3D 陣列 (uint8, 0-255)。
    """
    print(f"啟動分塊修復 (Grid: {splits}, Padding: {pad_size})...")

    # 1. 初始化輸出容器 (確保是 uint8，因為 execute_line_repair 回傳 uint8)
    patched_full = np.zeros(vol_3d.shape, dtype=np.uint8)

    # 2. 取得切分邏輯 (沿用之前的切分函數)
    base_slices = _octant_slices(vol_3d.shape, splits)

    for i, (sl_z, sl_y, sl_x) in enumerate(base_slices):
        # --- A. 計算原始座標 ---
        z_start, z_end = sl_z.start, sl_z.stop
        y_start, y_end = sl_y.start, sl_y.stop
        x_start, x_end = sl_x.start, sl_x.stop

        # --- B. 計算 Padding 後的座標 (限制在圖像範圍內) ---
        p_z_start = max(0, z_start)
        p_z_end = min(vol_3d.shape[0], z_end)
        p_y_start = max(0, y_start)
        p_y_end = min(vol_3d.shape[1], y_end)
        p_x_start = max(0, x_start)
        p_x_end = min(vol_3d.shape[2], x_end)

        # --- C. 取出子區塊 (含 Ghost Cells) ---
        sub_vol = vol_3d[p_z_start:p_z_end, p_y_start:p_y_end, p_x_start:p_x_end]

        # 這裡加個簡單的檢查，如果該區塊全是空的，就跳過運算以節省時間
        if not np.any(sub_vol):
            continue

        # --- D. 執行核心修復 ---
        # 注意：傳入子區塊進行運算。
        # 由於 execute_line_repair 內部會重新 label，這對於局部修復是正確的行為。
        # 只要 Padding 足夠，跨邊界的物件就能被視為連續。
        patched_sub = execute_line_repair(sub_vol, settings)

        # --- E. 裁切 (Remove Padding) ---
        # 計算相對於 sub_vol 的有效區域偏移量
        offset_z = z_start - p_z_start
        offset_y = y_start - p_y_start
        offset_x = x_start - p_x_start

        len_z = z_end - z_start
        len_y = y_end - y_start
        len_x = x_end - x_start

        valid_sub = patched_sub[
            offset_z: offset_z + len_z,
            offset_y: offset_y + len_y,
            offset_x: offset_x + len_x
        ]

        # --- F. 填入結果 ---
        patched_full[sl_z, sl_y, sl_x] = valid_sub

        # (可選) 顯示進度
        # print(f"  - Block {i+1}/{len(base_slices)} done.")

    return patched_full


def line_repair_msk(mask3d: np.ndarray, min_area: int = 10, max_link_dist: float = 30.0,
                    pass_iters: int = 1, line_thickness: int = 2):
    """
    核心修補邏輯：檢查 3D 連通物件是否在 2D 切片上斷裂並修復。
    """

    def label_3d_26(mask3d: np.ndarray):
        """26-connectivity 3D CC labeling."""
        structure = np.ones((3, 3, 3), dtype=bool)
        return ndi.label(mask3d.astype(bool), structure=structure)

    def label_2d_8(mask2d: np.ndarray):
        """8-connectivity 2D CC labeling."""
        structure = np.ones((3, 3), dtype=bool)
        return ndi.label(mask2d.astype(bool), structure=structure)

    def find_splits_in_slice(cc3d: np.ndarray, z: int, min_area: int = 10):
        """
        找出在 slice z 上：同一個 3D label 出現 >=2 個 2D 連通塊的情況。
        """
        lab2d = cc3d[z]
        Ls = np.unique(lab2d)
        Ls = Ls[Ls != 0]

        splits = []
        for L in Ls:
            m = (lab2d == L)
            if m.sum() < min_area: continue

            cc2d, n2d = label_2d_8(m)
            if n2d >= 2:
                # 過濾掉太小的 component，避免雜訊干擾連接
                areas = np.array([(cc2d == k).sum() for k in range(1, n2d + 1)])
                keep = np.where(areas >= min_area)[0] + 1
                if len(keep) >= 2:
                    splits.append((L, cc2d, keep))
        return splits

    def connect_components_by_nearest_points(mask2d: np.ndarray, cc2d: np.ndarray, comp_ids, line_thickness: int = 1):
        """
        在同一張 2D slice 中，針對指定的多塊 component 找最近點連線。
        """
        comps = [np.argwhere(cc2d == cid) for cid in comp_ids]
        if len(comps) < 2:
            return mask2d, None

        # 暴力找最近兩塊 (若點非常多可考慮 KDTree 優化，但在分割圖上通常還好)
        best = None
        best_pts = None

        # 簡化：只連接最近的一對，避免過度連接
        # 如果希望串聯所有斷開部分，需要改為 Minimum Spanning Tree 邏輯，但這裡先維持你原本的邏輯
        for i in range(len(comps)):
            for j in range(i + 1, len(comps)):
                A, B = comps[i], comps[j]
                # 計算兩兩距離矩陣
                d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)
                idx = np.unravel_index(np.argmin(d2), d2.shape)
                dist = np.sqrt(d2[idx])

                if best is None or dist < best:
                    best = dist
                    best_pts = (tuple(A[idx[0]]), tuple(B[idx[1]]))

        if best_pts is None: return mask2d, None

        (r0, c0), (r1, c1) = best_pts
        rr, cc = line(r0, c0, r1, c1)

        # 邊界檢查，防止 skimage.draw.line 超出範圍 (雖然理論上不應發生)
        valid = (rr >= 0) & (rr < mask2d.shape[0]) & (cc >= 0) & (cc < mask2d.shape[1])
        rr, cc = rr[valid], cc[valid]

        out = mask2d.copy()
        out[rr, cc] = True

        if line_thickness >= 2:
            line_only = np.zeros_like(mask2d, dtype=bool)
            line_only[rr, cc] = True
            structure = np.ones((3, 3), dtype=bool)
            # 膨脹線條使其變粗
            line_fat = ndi.binary_dilation(line_only, structure=structure, iterations=line_thickness - 1)
            out |= line_fat

        return out, (best, best_pts)

    mask = mask3d.astype(bool).copy()

    for _ in range(pass_iters):
        cc3d, n = label_3d_26(mask)
        changed = 0

        for z in range(mask.shape[0]):
            splits = find_splits_in_slice(cc3d, z, min_area=min_area)
            for (L, cc2d, comp_ids) in splits:
                repaired2d, info = connect_components_by_nearest_points(
                    mask[z], cc2d, comp_ids, line_thickness=line_thickness
                )
                if info is None: continue

                dist, _ = info
                if dist <= max_link_dist:
                    mask[z] = repaired2d
                    changed += 1

        if changed == 0:
            break

    return mask


# ==========================================
# 2. 現有後處理函數 (Auxiliary Functions)
# ==========================================


def normalize_segments_3d(vol_bool: np.ndarray, radius: float = 1.5, border: int = 8, axis: int = 0, sigma: float = 1.0,
                          repair: bool = False, max_dist: int = 30, max_angle_deg: int = 30,
                          safe_distance: int = 4, repair_dijkstra: bool = False, iterations: int = 1) -> np.ndarray:
    """
    將 3D 體積切成 8 塊後，對每塊分別進行線段正規化。
    axis: 0 為 Z 軸, 1 為 Y 軸, 2 為 X 軸
    """

    normalized_vol = vol_bool.copy()
    skel_vol = np.zeros_like(vol_bool)
    labeled_vol, num_features = ndi.label(normalized_vol, structure=ndi.generate_binary_structure(rank=3, connectivity=1))
    slices_list = _octant_slices(normalized_vol.shape, splits=(2, 2, 2))

    def get_2d_endpoints(skeleton):
        # 建立一個中間為 0，周圍為 1 的 kernel
        kernel = np.array([[1, 1, 1],
                           [1, 0, 1],
                           [1, 1, 1]])
        neighbor_count = convolve(skeleton.astype(int), kernel, mode='constant', cval=0)

        # 骨架點且鄰居只有 1 個
        endpoints = np.argwhere((skeleton == 1) & (neighbor_count == 1))
        return endpoints

    def trace_back_and_get_vector_2d(skeleton, endpoint, steps=3):
        """
        從端點沿著骨架往回走指定的步數，計算生長方向向量。
        """
        current_pt = endpoint
        visited = {tuple(endpoint)}

        for _ in range(steps):
            y, x = current_pt
            y_min, y_max = max(0, y - 1), min(skeleton.shape[0], y + 2)
            x_min, x_max = max(0, x - 1), min(skeleton.shape[1], x + 2)

            neighborhood = skeleton[y_min:y_max, x_min:x_max]
            neighbors = np.argwhere(neighborhood == 1)

            # 轉換為全局座標
            neighbors += np.array([y_min, x_min])

            next_pt = None
            for n in neighbors:
                if tuple(n) not in visited:
                    next_pt = n
                    break

            if next_pt is None:
                break  # 骨架太短，提早走到盡頭

            visited.add(tuple(next_pt))
            current_pt = next_pt

        # 向量方向：從回溯點指向端點 (即未來的生長趨勢)
        vec = endpoint - current_pt
        norm = np.linalg.norm(vec)
        if norm == 0:
            return np.array([0.0, 0.0])
        return vec / norm

    def match_endpoints_2d(endpoints, vectors, label_slice, max_dist=30, max_angle_deg=45):
        """
        根據距離和方向夾角配對端點
        """
        matched_pairs = []
        used_indices = set()
        max_angle_rad = math.radians(max_angle_deg)

        for i in range(len(endpoints)):
            if i in used_indices: continue

            best_match = -1
            min_dist = max_dist

            for j in range(len(endpoints)):
                if i == j or j in used_indices: continue

                p_A, v_A = endpoints[i], vectors[i]
                p_B, v_B = endpoints[j], vectors[j]

                label_A = label_slice[p_A[0], p_A[1]]
                label_B = label_slice[p_B[0], p_B[1]]

                if label_A != label_B:
                    continue

                # 1. 檢查距離
                dist = np.linalg.norm(p_A - p_B)
                if dist > max_dist: continue

                # 2. 檢查方向性
                vec_AB = (p_B - p_A) / dist
                vec_BA = -vec_AB

                # 確保向量長度不為 0 才計算夾角
                if np.linalg.norm(v_A) == 0 or np.linalg.norm(v_B) == 0: continue

                dot_A_forward = np.dot(v_A, vec_AB)
                dot_B_forward = np.dot(v_B, vec_BA)

                # 2. 檢查兩個端點本身的生長向量是否「相向」(例如夾角大於 135 度)
                # v_B 應該要和 v_A 大致反向，所以 v_A 和 -v_B 應該要大致同向
                cos_dirs = np.clip(np.dot(v_A, -v_B), -1.0, 1.0)
                dirs_aligned = math.acos(cos_dirs) < max_angle_rad  # 這裡的 max_angle_rad 可以設為 45 度 (即允許 45 度的方向誤差)

                if dot_A_forward > 0 and dot_B_forward > 0 and dirs_aligned:
                    if dist < min_dist:
                        min_dist = dist
                        best_match = j

            if best_match != -1:
                matched_pairs.append({
                    'pA': endpoints[i], 'pB': endpoints[best_match],
                    'vA': vectors[i], 'vB': vectors[best_match]
                })
                used_indices.add(i)
                used_indices.add(best_match)

        return matched_pairs

    def connect_skeleton_holes_2d(skeleton, label_slice, max_dist=30, max_angle_deg=45, trace_steps=5):
        """
        主函數：輸入 2D 骨架矩陣，輸出修補好的骨架矩陣
        """
        # 複製一份準備畫線用
        repaired_skeleton = skeleton.copy()

        # 1. 抓取端點
        raw_endpoints = get_2d_endpoints(skeleton)
        if len(raw_endpoints) < 2:
            return repaired_skeleton

        # 2. 【修改處】：直接先拿掉太短的骨架，不參與配對
        valid_endpoints = []
        valid_vectors = []
        for ep in raw_endpoints:
            # 這裡的 trace_steps 可以依需求設為 3 或 5
            vec = trace_back_and_get_vector_2d(skeleton, ep, steps=trace_steps)

            # 如果 vec 不是 None，代表這條骨架夠長，才允許加入配對池
            if vec is not None:
                valid_endpoints.append(ep)
                valid_vectors.append(vec)

        if len(valid_endpoints) < 2:
            return repaired_skeleton

        # 3. 執行配對
        matched_pairs = match_endpoints_2d(valid_endpoints, valid_vectors, label_slice, max_dist, max_angle_deg)

        # 4. Bresenham 畫線補洞
        for match in matched_pairs:
            pA, pB = match['pA'], match['pB']
            vA, vB = match['vA'], match['vB']
            # skimage.draw.line 會回傳線上所有點的 (y, x) 座標
            rr, cc = line(pA[0], pA[1], pB[0], pB[1])
            repaired_skeleton[rr, cc] = 1

        return repaired_skeleton

    def connect_skeleton_holes_dijkstra(skeleton, slice_2d, label_slice, max_dist=30, max_angle_deg=45, trace_steps=5):
        """
        主函數：利用 Dijkstra 與成本圖連接骨架
        """
        repaired_skeleton = skeleton.copy()

        # 1. 抓取端點
        raw_endpoints = get_2d_endpoints(skeleton)
        if len(raw_endpoints) < 2:
            return repaired_skeleton

        # 2. 直接先拿掉太短的骨架，不參與配對
        valid_endpoints = []
        valid_vectors = []
        for ep in raw_endpoints:
            vec = trace_back_and_get_vector_2d(skeleton, ep, steps=trace_steps)
            if vec is not None:
                valid_endpoints.append(ep)
                valid_vectors.append(vec)

        if len(valid_endpoints) < 2:
            return repaired_skeleton

        # 3. 執行配對
        matched_pairs = match_endpoints_2d(valid_endpoints, valid_vectors, label_slice, max_dist, max_angle_deg)
        if not matched_pairs:
            return repaired_skeleton

        # ==========================================
        # 【新增處】：利用連通集區分真正的目標區域與無關孤島
        # ==========================================
        # 取出當前 label 的所有區域
        current_mask_full = (slice_2d == label_slice)

        # 標記所有連通的孤島
        labeled_mask, num_features = ndi.label(current_mask_full)

        # 找出包含 valid_endpoints 的所有孤島 ID
        target_island_ids = set()
        for ep in valid_endpoints:
            island_id = labeled_mask[ep[0], ep[1]]
            if island_id != 0:
                target_island_ids.add(island_id)

        # 定義區域
        # is_target_mask: 只有包含端點的孤島才享有最低成本
        is_target_mask = np.isin(labeled_mask, list(target_island_ids))

        # is_disconnected_same_label: 同 label 但不含端點的孤島 (視同背景處理，避免被借道)
        is_disconnected_same_label = current_mask_full & (~is_target_mask)

        is_background = (slice_2d == 0) | is_disconnected_same_label
        is_other_mask = (~is_background) & (~is_target_mask)

        # ==========================================

        # 4. 建立成本圖 (Cost Map)
        cost_map = np.ones_like(slice_2d, dtype=np.float32)

        # --- A. 建立避開「其他 Mask」的護城河 ---
        if np.any(is_other_mask):
            dist_to_others = distance_transform_edt(~is_other_mask)
            max_warning_penalty = 15.0

            warning_penalty = np.clip(max_warning_penalty * (1 - dist_to_others / safe_distance), 0,
                                      max_warning_penalty)
            cost_map += warning_penalty

        # --- B. 設定各地形基礎成本 ---

        # 1. 自己的目標 Mask 內部：保持最低成本 1.0
        # (cost_map[is_target_mask] 維持原樣)

        # 2. 黑色背景與「無關的同 label 孤島」：給予基礎懲罰
        bg_penalty = 2.0  # 建議稍微提高背景懲罰，強迫走最短直線
        cost_map[is_background] = bg_penalty

        # 3. 其他 Mask：絕對高牆
        cost_map[is_other_mask] = 1e6

        # 5. 使用 Dijkstra 畫線補洞
        for match in matched_pairs:
            pA, pB = tuple(match['pA']), tuple(match['pB'])

            try:
                path, cost = route_through_array(cost_map, pA, pB, fully_connected=True)

                if cost >= max_dist * bg_penalty * 2:  # 注意這裡的閾值可能需要配合 bg_penalty 調整
                    continue

                path_y = [p[0] for p in path]
                path_x = [p[1] for p in path]

                repaired_skeleton[path_y, path_x] = 1
            except ValueError:
                continue

        return repaired_skeleton

    def _process_block(block: np.ndarray, labeled_block: np.ndarray, radius: float, border: int,
                       axis: int, sigma: float = 0.0) -> tuple[np.ndarray, np.ndarray]:

        block_out = block.copy()
        skel_out = block.copy()
        shape = block.shape

        # 1. 檢查非處理軸的其餘兩個維度是否夠大
        other_dims = [shape[i] for i in range(3) if i != axis]
        if border > 0 and any(d <= 2 * border for d in other_dims):
            return block_out, skel_out  # 修正：確保提早結束時也回傳兩個陣列

        # 2. 定義統一的內部區域切片邊界
        inner_slice = slice(border, -border if border > 0 else None)

        # 3. 沿著指定的 axis 進行迭代
        for i in range(shape[axis]):
            # 動態生成 3D 讀取切片 (例如 axis=1, i=5 時，等同於 [:, 5, :])
            read_idx = [slice(None)] * 3
            read_idx[axis] = i
            read_idx = tuple(read_idx)

            slice_2d = block[read_idx]
            if not np.any(slice_2d):
                continue

            label_slice_2d = labeled_block[read_idx]
            skel = skeletonize(slice_2d, method='lee')
            if repair:
                for it in range(iterations):
                    if repair_dijkstra:
                        skel = connect_skeleton_holes_dijkstra(skeleton=skel, slice_2d=slice_2d,
                                                               label_slice=label_slice_2d,
                                                               max_dist=max_dist, max_angle_deg=max_angle_deg)
                    else:
                        skel = connect_skeleton_holes_2d(skeleton=skel, label_slice=label_slice_2d, max_dist=max_dist,
                                                         max_angle_deg=max_angle_deg)

            if np.any(skel):
                # 距離變換擴張
                reconstructed = distance_transform_edt(~skel) <= radius

                # 加入 2D 高斯模糊
                if sigma > 0:
                    reconstructed = gaussian_filter(reconstructed.astype(float), sigma=sigma)
                    reconstructed = reconstructed > 0.3

                # 動態生成 3D 寫回切片 (保留 border，並指定當前層 i)
                write_idx = [inner_slice] * 3
                write_idx[axis] = i
                write_idx = tuple(write_idx)

                # reconstructed 是 2D，我們只需要擷取它內部的部分來填入 3D 結構
                recon_idx = tuple([inner_slice] * 2)

                block_out[write_idx] = reconstructed[recon_idx]
                skel_out[write_idx] = skel[recon_idx]

        return block_out, skel_out

    # 疊代處理 8 個子塊
    for sl in slices_list:
        sub_vol = vol_bool[sl]
        labeled_sub_vol = labeled_vol[sl]
        processed_sub_vol, skel_sub_vol = _process_block(sub_vol, labeled_sub_vol, radius, border, axis)
        normalized_vol[sl] = processed_sub_vol
        skel_vol[sl] = skel_sub_vol

    return normalized_vol, skel_vol



def normalize_segments_3d_normal(vol_bool: np.ndarray, radius: float = 1.5, border: int = 8) -> np.ndarray:
    """
        逐 Z 軸切片進行線段正規化：先骨架化再擴張。

        參數:
        - vol_bool: 輸入的 3D 布林陣列 (D, H, W)
        - radius: 擴張半徑，用於統一線段粗細
        """
    # 建立輸出的容器，預設為全 False (或根據需求複製原圖)
    normalized_vol = np.zeros_like(vol_bool, dtype=bool)
    depth = vol_bool.shape[0]

    for z in range(depth):
        slice_2d = vol_bool[z, :, :]

        # 如果該層沒有任何像素，直接跳過
        if not np.any(slice_2d):
            continue

        # 1. 骨架化：將線條縮減為 1 像素寬度的中心線
        # 注意：輸入必須是布林值
        skel = skeletonize(slice_2d)

        # 2. 擴張 (使用距離變換實現精準半徑控制)
        if np.any(skel):
            # 計算每個背景點到最近骨架點的距離
            dist_map = distance_transform_edt(~skel)
            # 距離在半徑內的點即為新的線段區域
            reconstructed_slice = dist_map <= radius

            # 3. 存入結果
            normalized_vol[z, :, :] = reconstructed_slice

    return normalized_vol


def apply_y_axis_closing(vol, iterations=1):
    """Y 軸方向閉運算"""
    processed_vol = np.copy(vol)
    for y in range(vol.shape[1]):
        slice_2d = vol[:, y, :]
        if np.any(slice_2d):
            closed_slice = binary_closing(slice_2d, iterations=iterations)
            processed_vol[:, y, :] = closed_slice
    return processed_vol


def apply_z_axis_fill_holes(vol):
    """Z 軸方向孔洞填充"""
    processed_vol = np.copy(vol)
    for z in range(vol.shape[0]):
        slice_2d = vol[z, :, :]
        if np.any(slice_2d):
            filled_slice = binary_fill_holes(slice_2d)
            processed_vol[z, :, :] = filled_slice
    return processed_vol


def apply_y_axis_fill_holes(vol):
    """Y 軸方向孔洞填充"""
    processed_vol = np.copy(vol)
    for y in range(vol.shape[1]):
        slice_2d = vol[:, y, :]
        if np.any(slice_2d):
            filled_slice = binary_fill_holes(slice_2d)
            processed_vol[:, y, :] = filled_slice
    return processed_vol


import numpy as np


def robust_mask_sandwich(mask, max_gap=2, axis_weights=[0, 1, 1], min_neighbors=2):
    """
    改良版 Mask Sandwich (加入 has_support 檢查):

    Args:
        mask: 3D Binary Mask
        max_gap: 最大填補間隙
        axis_weights: [z, y, x] 開關。
        min_neighbors: 一個像素周圍 (3x3範圍內) 至少要有幾個鄰居才算「結實」。
                       建議設為 2 或 3。如果設為 0 則等於沒檢查。
    """
    refined = mask.copy()
    ndim = 3

    # 定義檢查函數：輸入一個 2D slice，回傳「結實像素」的 Mask
    for axis in range(ndim):
        if axis_weights[axis] == 0:
            continue

        current_max_gap = max_gap

        for gap in range(1, current_max_gap + 1):
            stride = gap + 1

            s_prev = [slice(None)] * ndim
            s_next = [slice(None)] * ndim

            s_prev[axis] = slice(0, -stride)
            s_next[axis] = slice(stride, None)

            # 1. 取出兩端切片
            slice_prev = refined[tuple(s_prev)]
            slice_next = refined[tuple(s_next)]

            # 2. 【核心修改】檢查支撐性 (Support Check)
            # 這裡我們只對「當前操作面」做 2D 檢查
            # 注意：slice_prev 和 slice_next 可能是 3D 的 (一部分的 volume)
            # 為了效能，我們可以批次處理或簡化處理

            # 如果是處理 Z 軸，slice_prev 是 (D', H, W)，我們希望在 H,W 平面檢查
            # 如果是處理 Y 軸，slice_prev 是 (D, H', W)，我們希望在 D,W 平面檢查 (有點怪)
            # 但通常「噪點」定義在 3D 空間都是通用的。
            # 為了簡化且通用，我們直接比較兩端是否重疊，
            # 並利用 Logical AND 的特性：噪音通常不會剛好在隔壁層的同個位置

            # --- 實作支撐性過濾 ---
            # 為了避免複雜的軸向判斷，這裡做一個取捨：
            # 我們假設輸入的 slice 已經是二值化，直接計算鄰居太慢？
            # 不會，scipy convolve 在 CPU 上對 binary mask 很快。

            # 針對不同的軸向，我們需要正確的 Kernel
            # 為了通用性，我們在函數內動態構建 N-dim Kernel 比較慢
            # 但考慮到 Vesuvius 的各向異性，我們主要關心 "XY平面" 的鄰居

            if axis == 0:  # 正在修補 Z 軸間隙，檢查 XY 平面的鄰居
                # 這裡 slice_prev 的形狀是 (D_subset, H, W)
                # 我們可以對每一層做 2D 卷積，或者直接用 3D 卷積但 Kernel 只有 XY 有值

                # 建立 3D Kernel 但只在 XY 平面擴展 (3x3x1 concept)
                kernel_3d = np.zeros((3, 3, 3), dtype=np.uint8)
                kernel_3d[1, :, :] = 1  # 中間層的 3x3
                kernel_3d[1, 1, 1] = 0  # 扣掉自己

                # 計算支撐
                count_prev = convolve(slice_prev.astype(np.uint8), kernel_3d, mode='constant', cval=0)
                count_next = convolve(slice_next.astype(np.uint8), kernel_3d, mode='constant', cval=0)

                valid_prev = (slice_prev) & (count_prev >= min_neighbors)
                valid_next = (slice_next) & (count_next >= min_neighbors)

            else:
                # 針對 Y 或 X 軸填補，通常比較少用，或者可以直接忽略支撐檢查(設寬鬆)
                # 或者簡單一點：只要有值就算 (不做額外檢查)，因為 XY 斷裂修復通常比較安全
                valid_prev = slice_prev
                valid_next = slice_next
                # 如果你想非常嚴格，也可以在這裡實作對應軸的 convolve，但代碼會變很長

            # 3. 結合條件：兩端都有值 + 兩端都結實
            bridge_candidates = valid_prev & valid_next

            if not np.any(bridge_candidates):
                continue

            # 4. 執行填補
            for i in range(1, stride):
                s_fill = [slice(None)] * ndim
                if -stride + i == 0:
                    s_fill[axis] = slice(i, None)
                else:
                    s_fill[axis] = slice(i, -stride + i)

                refined[tuple(s_fill)] |= bridge_candidates

    return refined


def robust_mask_sandwich_v3(mask, max_gap=2, axis_weights=[1, 1, 1], min_neighbors=2, kernel_size=5):
    """
    V3 優化強健版（支援擴大 Kernel）：
    1. 預計算支撐圖 (Support Map)，大幅提升效能。
    2. 全軸向支持：所有軸向均可進行 2D 支撐檢查。
    3. 記憶體優化：最小化陣列複製次數。
    4. 動態 Kernel：可透過 kernel_size 參數（需為奇數）擴大鄰居檢查範圍。
    """
    if kernel_size % 2 == 0:
        raise ValueError("kernel_size 必須為奇數（例如 3, 5, 7）")

    # 統一轉換為 bool 進行運算，減少記憶體壓力
    refined = mask.astype(bool, copy=True)
    ndim = 3
    half_k = kernel_size // 2

    # --- 預計算：每一層的支撐圖 (只算一次) ---
    # 這樣在後續遍歷不同的 gap 時，不需要重複捲積
    support_masks = []
    for axis in range(ndim):
        if axis_weights[axis] == 0:
            support_masks.append(None)
            continue

        # 建立該軸向的 2D 鄰居統計 Kernel
        # 若 kernel_size=5，則 k_shape 預設為 [5, 5, 5]
        k_shape = [kernel_size] * ndim
        k_shape[axis] = 1  # 讓 Kernel 在該軸向上是扁平的 (例如 [1, 5, 5])
        kernel = np.ones(k_shape, dtype=np.uint8)

        # 移除中心點 (不計算自己)
        # 動態定位中心，例如 kernel_size=5，half_k=2，center=[2, 2, 2]
        center = [half_k] * ndim
        center[axis] = 0
        kernel[tuple(center)] = 0

        # 一次性計算整體的鄰居數量
        neighbor_count = convolve(refined.astype(np.uint8), kernel, mode='constant', cval=0)

        # 只有「自己是1」且「鄰居夠多」的才算有效錨點
        support_masks.append((refined) & (neighbor_count >= min_neighbors))

    # --- 執行填補 ---
    for axis in range(ndim):
        if axis_weights[axis] == 0:
            continue

        valid_map = support_masks[axis]
        shape_at_axis = refined.shape[axis]

        for gap in range(1, max_gap + 1):
            stride = gap + 1

            # 取得兩端的「強健錨點」
            s_prev = [slice(None)] * ndim
            s_next = [slice(None)] * ndim
            s_prev[axis] = slice(0, -stride)
            s_next[axis] = slice(stride, None)

            # 只有兩端都是「結實像素」時，才建立橋樑
            bridge = valid_map[tuple(s_prev)] & valid_map[tuple(s_next)]

            if not np.any(bridge):
                continue

            # 填補中間空隙
            for i in range(1, stride):
                s_fill = [slice(None)] * ndim
                s_fill[axis] = slice(i, i + (shape_at_axis - stride))
                refined[tuple(s_fill)] |= bridge

    return refined


def robust_sandwich_fill(mask, max_gap=2, fix_planes=('XZ', 'YZ'), kernel_size=3, min_neighbors=2):
    """
    支援大空隙 (max_gap) 的對角線強健版填補（動態更新錨點模式）。
    每次填補後會即時更新可用的「強健錨點」，允許產生連鎖填補效應。
    """
    if kernel_size % 2 == 0:
        raise ValueError("kernel_size 必須為奇數（例如 3, 5, 7）")

    # 統一轉換為 bool 進行運算，減少記憶體壓力
    img = mask.astype(bool, copy=True)
    refined = img.copy()
    ndim = 3
    shape = refined.shape

    # 準備 3D 鄰居統計 Kernel (若需要判斷鄰居才會用到)
    if min_neighbors > 0:
        support_kernel = np.ones((kernel_size, kernel_size, kernel_size), dtype=np.uint8)
        half_k = kernel_size // 2
        support_kernel[half_k, half_k, half_k] = 0

    # 定義要修復的平面對角線方向 (dz, dy, dx)
    plane_map = {
        'XZ': [(1, 0, 1), (1, 0, -1)],
        'YZ': [(1, 1, 0), (1, -1, 0)],
        'XY': [(0, 1, 1), (0, 1, -1)]
    }

    selected_dirs = []
    for p in fix_planes:
        if p in plane_map:
            selected_dirs.extend(plane_map[p])

    # 執行切片位移對角線填補
    for dz, dy, dx in selected_dirs:
        dir_vector = (dz, dy, dx)

        for gap in range(1, max_gap + 1):

            # --- 【核心修改】：動態更新錨點 ---
            # 每次處理新方向或新間距前，根據「最新的 refined 狀態」重新計算錨點
            if min_neighbors > 0:
                # 重新執行卷積以獲取最新鄰居數量
                neighbor_count = convolve(refined.astype(np.uint8), support_kernel, mode='constant', cval=0)
                valid_anchors = refined & (neighbor_count >= min_neighbors)
            else:
                # 效能優化：如果 min_neighbors=0，最新的錨點就等於當前的 refined 狀態，免算卷積
                valid_anchors = refined.copy()
            # ----------------------------------

            stride = gap + 1
            stride_offset = tuple(d * stride for d in dir_vector)

            s_base = []
            s_target = []
            lengths = []
            starts = []

            for a in range(ndim):
                offset_a = stride_offset[a]
                if offset_a > 0:
                    s_base.append(slice(0, -offset_a))
                    s_target.append(slice(offset_a, None))
                    lengths.append(shape[a] - offset_a)
                    starts.append(0)
                elif offset_a < 0:
                    s_base.append(slice(-offset_a, None))
                    s_target.append(slice(0, offset_a))
                    lengths.append(shape[a] - abs(offset_a))
                    starts.append(-offset_a)
                else:
                    s_base.append(slice(None))
                    s_target.append(slice(None))
                    lengths.append(shape[a])
                    starts.append(0)

            # 找出兩端都是「最新強健錨點」的橋樑
            bridge = valid_anchors[tuple(s_base)] & valid_anchors[tuple(s_target)]

            if not np.any(bridge):
                continue

            # 將橋樑中間的「所有」空隙點補齊
            for i in range(1, stride):
                fill_offset = tuple(d * i for d in dir_vector)
                s_fill = []

                for a in range(ndim):
                    start = starts[a] + fill_offset[a]
                    end = start + lengths[a]
                    s_fill.append(slice(start, end))

                # 更新 refined，供下一個迴圈使用
                refined[tuple(s_fill)] |= bridge

    return refined


def xyz_multi_gap_fill_v2(mask, max_gap=2):
    """
    優化版：使用向量化位移檢查三明治結構。
    """
    refined = mask.astype(bool, copy=True)

    for axis in range(3):
        # 針對每一種可能的 gap 長度進行檢查
        for gap in range(1, max_gap + 1):
            stride = gap + 1

            # 建立位移切片
            # 這裡利用 numpy 的 slice 技巧，找出距離為 stride 的兩端
            s_prev = [slice(None)] * 3
            s_next = [slice(None)] * 3
            s_fill_base = [slice(None)] * 3

            s_prev[axis] = slice(0, -stride)
            s_next[axis] = slice(stride, None)

            # 找出兩端皆為 1 的橋樑錨點
            bridge = refined[tuple(s_prev)] & refined[tuple(s_next)]

            # 填補中間的所有像素
            for i in range(1, stride):
                s_fill = list(s_fill_base)
                s_fill[axis] = slice(i, i + (refined.shape[axis] - stride))
                refined[tuple(s_fill)] |= bridge

    return refined


def diagonal_sandwich_fill_v2(mask, fix_planes):
    """
        精確版：只針對特定平面進行對角線修復，避免過度填補。
        """
    img = mask.astype(np.uint8)
    refined = img.copy()

    plane_map = {
        'XZ': [(1, 0, 1), (1, 0, -1)],
        'YZ': [(1, 1, 0), (1, -1, 0)],
        'XY': [(0, 1, 1), (0, 1, -1)]
    }

    selected_directions = []
    for p in fix_planes:
        if p in plane_map:
            print(f'Fixing {p} plane.')
            selected_directions.extend(plane_map[p])

    for dz, dy, dx in selected_directions:
        kernel = np.zeros((3, 3, 3), dtype=np.uint8)
        kernel[1 + dz, 1 + dy, 1 + dx] = 1
        kernel[1 - dz, 1 - dy, 1 - dx] = 1

        # 卷積檢查
        counts = convolve(refined, kernel, mode='constant', cval=0)
        refined[counts == 2] = 1

    return refined > 0


def fill_3d_projection_holes_blocked(binary_mask):
    """
    將 binary_mask 切成 8 塊 (2x2x2) 後分別進行投影補洞，最後合併。
    """

    def process_single_block(sub_mask):
        """
        處理單個 3D 子區塊的核心邏輯。
        包含：移除小物件、骨架檢查、投影填補。
        """
        # 1. 複製並過濾小物件
        # 注意：這裡的 min_size 作用於「子區塊內的物件體積」。
        # 若物件被切開，其局部體積可能小於 5000，會被暫時過濾掉 (但最後會通過原始 mask 合併回來)。
        mask_filtered = remove_small_objects(sub_mask, min_size=5000, connectivity=1)

        # 用於運算的 output，基於過濾後的 mask 進行修改
        processing_mask = mask_filtered.copy()

        # 2. 提取 3D 連通域
        struct_3d = ndimage.generate_binary_structure(3, 1)
        labeled_array, num_features = ndimage.label(mask_filtered, structure=struct_3d)

        # 用於檢查 2D (ZY平面) 連接關係的結構 (4 連通)
        struct_2d = ndimage.generate_binary_structure(2, 2)

        # 若沒有大型物件，直接回傳全 false 的 processing_mask (其實就是空的)
        if num_features == 0:
            return processing_mask

        # 3. 遍歷大型連通域
        for i in range(1, num_features + 1):
            slices = ndimage.find_objects(labeled_array == i)[0]
            comp_mask = (labeled_array[slices] == i)

            # ==========================================================
            # 骨架化檢查
            # ==========================================================
            total_pixels = np.sum(comp_mask)
            if total_pixels == 0: continue

            width_x = comp_mask.shape[2]
            skeleton_pixels = 0

            # 針對 X 軸的每個 2D 切片做骨架化
            for x in range(width_x):
                current_slice_zy = comp_mask[:, :, x]
                if np.any(current_slice_zy):
                    skel_slice = skeletonize(current_slice_zy)
                    skeleton_pixels += np.sum(skel_slice)

            ratio = skeleton_pixels / total_pixels
            if ratio > 0.1:  # 骨架佔比過高，跳過
                continue
            # ==========================================================

            # 4. 投影與填補邏輯
            projection_zy = np.any(comp_mask, axis=2)
            filled_projection = ndimage.binary_fill_holes(projection_zy)
            hole_mask_zy = filled_projection ^ projection_zy

            if not np.any(hole_mask_zy):
                continue

            hole_dilated = ndimage.binary_dilation(hole_mask_zy, structure=struct_2d)

            x_start = slices[2].start
            for x in range(width_x):
                current_slice_zy = comp_mask[:, :, x]
                is_connected = np.any(hole_dilated & current_slice_zy)

                if is_connected:
                    # 修改該區塊的 processing_mask
                    target_slice = processing_mask[slices[0], slices[1], x_start + x]
                    target_slice[hole_dilated] = True

        return processing_mask

    # 確保輸入是布林值
    mask = binary_mask.astype(bool)

    # 1. 取得切片清單 (2x2x2 = 8塊)
    splits = (2, 2, 2)
    slices_list = _octant_slices(mask.shape, splits)

    # 建立一個全域的空陣列，用來存放處理過(填補後)的大物件
    processed_global = np.zeros_like(mask)

    print(f"開始分塊處理：將影像切分為 {len(slices_list)} 塊...")

    # 2. 迭代每個區塊
    for idx, (sz, sy, sx) in enumerate(slices_list):
        # 取出子區塊
        sub_mask = mask[sz, sy, sx]

        # 如果子區塊全空，跳過運算
        if not np.any(sub_mask):
            continue

        # 執行核心補洞邏輯
        sub_result = process_single_block(sub_mask)

        # 將結果放回全域陣列
        processed_global[sz, sy, sx] = sub_result

    # 3. 合併結果
    # final_output = 原始小物件 (從原始 mask 保留) | 處理後的大物件與填補 (從 processed_global 取得)
    final_output = mask | processed_global

    return final_output


def compute_separation_constraint(mask):
    """
    計算連通域間的不可侵犯領域與邊界。

    規則：
    1. 定義物件：使用 6 連通 (Face-connected) 區分物件。
    2. 劃分領土：使用 Voronoi (EDT) 擴張，填滿背景。
    3. 定義邊界：若體素的 26 連通鄰域 (3x3x3) 內包含 ">=2 個不同的連通域 ID"，則視為邊界。
    """

    # -----------------------------------------------------------
    # 1. 計算 3D 連通域 (Labeling) - 嚴格區分對角線接觸
    # -----------------------------------------------------------
    mask = mask.copy()
    mask = clear_boundary_faces(mask, margin=3)

    mask = remove_small_objects(mask > 0, 5000)
    # 使用 6 連通 (connectivity=1)，確保僅在對角線接觸的物件被視為不同個體
    structure_6conn = ndimage.generate_binary_structure(rank=3, connectivity=1)
    labeled_array, num_features = ndimage.label(mask, structure=structure_6conn)

    # 如果全場只有 0 或 1 個物件，不存在"交界"，直接回傳全 True
    if num_features <= 1:
        return np.ones_like(mask, dtype=bool)

    # -----------------------------------------------------------
    # 2. 擴張勢力範圍 (Voronoi / EDT)
    # -----------------------------------------------------------
    # 我們需要一個"填滿"的空間圖，每個點都知道自己最近的物件是誰 (Territory ID)
    # 對"背景 (0)"做距離變換，找到最近的前景索引
    # indices shape: (3, D, H, W)
    _, indices = ndimage.distance_transform_edt(labeled_array == 0, return_indices=True, return_distances=True)

    # 映射回 Label ID，得到全空間的領土圖
    # 這裡 territory 不再有 0 (背景)，全都是 1~N 的 ID
    territory = labeled_array[indices[0], indices[1], indices[2]]

    # -----------------------------------------------------------
    # 3. 標記交界處 (26-Connectivity Multi-Label Detection)
    # -----------------------------------------------------------
    # 規則：如果一個體素的 26 連通鄰域內，包含 2 個以上的不同 ID，它就是邊界。
    # 數學實作：在 territory map 上，若 Max(鄰域) != Min(鄰域)，則必有至少兩個不同 ID。

    footprint_26conn = np.ones((3, 3, 3), dtype=int)  # 26 連通核心

    # 找出鄰域內的最大 ID 與最小 ID
    max_labels = ndimage.maximum_filter(territory, footprint=footprint_26conn)
    min_labels = ndimage.minimum_filter(territory, footprint=footprint_26conn)

    # 邊界判定：只要最大值不等於最小值，代表鄰域內混雜了不同的領土
    # 這會偵測到：
    # 1. 兩個物件擴張後的接觸面 (Voronoi 邊界)
    # 2. 兩個原始物件物理上靠得很近或接觸的地方 (Original Contact)
    boundary_mask = (max_labels != min_labels)

    # -----------------------------------------------------------
    # 4. 取得最終 Valid Mask
    # -----------------------------------------------------------
    # 只要不是邊界，就是合法填充區
    valid_mask = ~boundary_mask

    return valid_mask


def robust_mask_refine(mask, max_gap=2, iterations=1, min_neighbors=2, kernel_size=5):
    """
    包含分離保護的綜合修復流程。
    """
    # --- 前處理：計算不可侵犯領域 ---
    # print("Calculating separation constraints...")
    # valid_constraint = compute_separation_constraint(mask)

    res = mask.copy()

    # 開始修復迴圈
    for i in range(iterations):
        # 假設這是您的修復函數 (需確保您有定義這些函數)
        # res = xyz_multi_gap_fill_v2(res, max_gap=max_gap)

        # 1. 執行填充/修復
        # 注意：這裡假設您的 sandwich 或 diagonal 函數會讓 mask 變大
        res = robust_mask_sandwich_v3(res, max_gap=max_gap, min_neighbors=min_neighbors, kernel_size=kernel_size)
        res = robust_sandwich_fill(res, max_gap=1, min_neighbors=0, kernel_size=3,
                                   fix_planes=('XZ', 'YZ'))

        # 2. 應用不可侵犯領域約束
        # 強制切斷跨越連通域邊界的填充
    # res = res & valid_constraint
    return res


import numpy as np


def robust_mask_refine_8slice(mask, max_gap=2, iterations=1, min_neighbors=2, splits=(2, 2, 2)):
    """
    將 3D mask 切分為多個區塊（預設 2x2x2=8 份）平行或依序處理，
    並處理邊界重疊以保持修復的連續性。

    Args:
        mask (np.ndarray): 原始 3D Boolean/Int Mask。
        max_gap (int): 最大填補間隙 (傳遞給 robust_mask_refine)。
        iterations (int): 迭代次數 (傳遞給 robust_mask_refine)。
        min_neighbors (int): 最小鄰居數 (傳遞給 robust_mask_refine)。
        splits (tuple): (z_parts, y_parts, x_parts) 指定各軸切分數量，預設為 (2, 2, 2) 即 8 等份。

    Returns:
        np.ndarray: 修復完成的完整 Mask。
    """
    # 1. 準備輸出的容器
    refined_full = np.zeros_like(mask)

    # 2. 取得基礎切片 (根據 splits 數量切分)
    # 假設 _octant_slices 傳回 [(slice_z, slice_y, slice_x), ...]
    base_slices = _octant_slices(mask.shape, splits)

    for sl_z, sl_y, sl_x in base_slices:
        # --- A. 直接取出子區塊 (無 Padding) ---
        sub_mask = mask[sl_z, sl_y, sl_x]

        # --- B. 執行核心修復演算法 ---
        # 直接在子區塊上運算
        refined_sub = robust_mask_refine(
            sub_mask,
            max_gap=max_gap,
            iterations=iterations,
            min_neighbors=min_neighbors
        )

        # --- C. 直接填回結果陣列 ---
        # 因為沒有 Padding，不需要計算 offset 或裁切
        refined_full[sl_z, sl_y, sl_x] = refined_sub

    return refined_full


# def robust_mask_refine(mask, max_gap=2, max_iterations=15, min_neighbors=2):
#     """
#     綜合修復流程：結合軸向與對角線填充。
#     機制：若 res 經過處理後沒有變化就停止，否則最大執行 max_iterations 次。
#     輸出：會列印最終執行的次數。
#     """
#     res = mask.copy()
#
#     for i in range(max_iterations):
#         prev_res = res.copy()
#
#         # 執行修復操作
#         # res = xyz_multi_gap_fill_v2(res, max_gap=max_gap)
#         res = robust_mask_sandwich_v3(res, max_gap=max_gap, min_neighbors=min_neighbors)
#         res = diagonal_sandwich_fill_v2(res, fix_planes=('XZ', 'YZ'))
#
#         # 檢查是否收斂：若處理前後結果相同，則提早停止
#         if np.array_equal(res, prev_res):
#             # i 是從 0 開始，所以實際次數是 i + 1
#             print(f"處理已收斂，共執行了 {i + 1} 次。")
#             break
#     else:
#         # Python 的 for-else 語法：只有當迴圈「沒有」被 break 中斷（即跑滿 max_iterations）時才會執行
#         print(f"已達到最大設定次數，共執行了 {max_iterations} 次。")
#
#     return res

def _axis_cuts(n: int, parts: int) -> List[Tuple[int, int]]:
    """將維度 n 切分為 parts 份，處理餘數確保覆蓋全域。"""
    parts = max(1, min(parts, n if n > 0 else 1))
    base = n // parts
    extra = n % parts
    cuts = []
    start = 0
    for i in range(parts):
        size = base + (1 if i < extra else 0)
        end = start + size
        cuts.append((start, end))
        start = end
    return cuts


def _octant_slices(shape: Tuple[int, int, int], splits: Tuple[int, int, int]) -> List[Tuple[slice, slice, slice]]:
    """生成 3D 切片物件清單。"""
    zcuts = _axis_cuts(shape[0], splits[0])
    ycuts = _axis_cuts(shape[1], splits[1])
    xcuts = _axis_cuts(shape[2], splits[2])
    out = []
    for z0, z1 in zcuts:
        for y0, y1 in ycuts:
            for x0, x1 in xcuts:
                if (z1 - z0) > 0 and (y1 - y0) > 0 and (x1 - x0) > 0:
                    out.append((slice(z0, z1), slice(y0, y1), slice(x0, x1)))
    return out


def remove_small_8slice(vol, min_size=100, connectivity=1):
    """使用 8-slice 邏輯平行化/分段移除小物件。"""
    vol_bool = vol > 0
    output = np.zeros_like(vol_bool)

    # 取得 8 個區塊的 slice (2x2x2)
    slices = _octant_slices(vol_bool.shape, (2, 2, 2))
    for slc in slices:
        block = vol_bool[slc]
        if np.any(block):
            # 處理區塊並塞回對應位置
            output[slc] = remove_small_objects(
                block,
                min_size=min_size,
                connectivity=connectivity
            )

    return output.astype(np.uint8)


import numpy as np
import torch
import scipy.ndimage as ndimage


def reconstruct_surface_from_mask(
        mask_3d: np.ndarray,
        degree: int = 6,
        min_pixels: int = 20,
        device: str = 'cpu'
) -> np.ndarray:
    """
    輸入一個 3D Binary Mask，對每個 6-連通區域進行多項式曲面擬合重建。

    參數:
    - mask_3d: 輸入的 3D numpy array (0 或 1)
    - degree: 多項式的階數 (建議 4-8)
    - min_pixels: 忽略小於此像素數量的連通域
    - device: 'cpu' 或 'cuda'

    回傳:
    - output_mask: 重建後的 3D mask (僅包含擬合出的曲面，非實心體積)
    """

    # 1. 定義 6-連通結構 (3x3x3)
    # structure 設為 1 代表只有上下左右前後相連 (6-connectivity)
    s = ndimage.generate_binary_structure(3, 1)

    # 2. 標記連通域
    labeled_array, num_features = ndimage.label(mask_3d, structure=s)

    # 準備輸出容器
    output_mask = np.zeros_like(mask_3d)

    if num_features == 0:
        return output_mask

    print(f"檢測到 {num_features} 個連通域，開始處理...")

    # 3. 逐一處理每個連通域
    for label_id in range(1, num_features + 1):
        # 提取當前組件的點雲
        component_mask = (labeled_array == label_id)
        points_zyx = np.argwhere(component_mask)  # 格式: [z, y, x]

        # 忽略過小的噪點
        if len(points_zyx) < min_pixels:
            continue

        # 呼叫擬合核心函數
        reconstructed_points = _fit_poly_component(points_zyx, degree, device)

        # 4. 將重建的點雲填回輸出 Mask
        if reconstructed_points is not None:
            # 過濾掉超出原始邊界的點
            D, H, W = mask_3d.shape
            z, y, x = reconstructed_points[:, 0], reconstructed_points[:, 1], reconstructed_points[:, 2]

            valid_mask = (
                    (z >= 0) & (z < D) &
                    (y >= 0) & (y < H) &
                    (x >= 0) & (x < W)
            )

            valid_z = z[valid_mask].astype(int)
            valid_y = y[valid_mask].astype(int)
            valid_x = x[valid_mask].astype(int)

            # 填入輸出 (設為 1)
            output_mask[valid_z, valid_y, valid_x] = 1

    return output_mask


def _fit_poly_component(points_np, degree, device):
    """
    單一組件的 PCA + 多項式擬合核心邏輯 (基於 hengck23 的思路)
    """
    try:
        points = torch.tensor(points_np, dtype=torch.float64, device=device)

        # --- 1. PCA 座標轉換 (Alignment) ---
        mean = points.mean(dim=0)
        centered = points - mean

        # 使用 PCA 找到主軸 (SVD 分解)
        # V 的最後一列通常是法向量方向 (變異最小軸)，我們將其視為新的 Z 軸
        U, S, V = torch.pca_lowrank(centered, q=3)

        # 旋轉到 PCA 空間: [x', y', z']
        # 注意: 這裡我們假設前兩個主成分是展延面 (x, y)，第三個是高度 (z)
        pca_points = centered @ V

        x_pca = pca_points[:, 0]
        y_pca = pca_points[:, 1]
        z_pca = pca_points[:, 2]  # 這是我們要擬合的目標高度

        # 歸一化以避免數值不穩定
        x_scale = x_pca.abs().max() + 1e-6
        y_scale = y_pca.abs().max() + 1e-6

        # --- 2. 建構多項式矩陣 (Vandermonde Matrix) ---
        # 擬合目標: z_pca = Poly(x_pca, y_pca)
        A_list = []
        for i in range(degree + 1):
            for j in range(degree + 1 - i):
                term = ((x_pca / x_scale) ** i) * ((y_pca / y_scale) ** j)
                A_list.append(term)

        A = torch.stack(A_list, dim=1)

        # --- 3. 求解係數 (Least Squares) ---
        # Ridge Regression (加上 lambda 避免奇異矩陣)
        lam = 1e-3
        I = torch.eye(A.shape[1], device=device, dtype=torch.float64)
        coeffs = torch.linalg.solve(A.T @ A + lam * I, A.T @ z_pca)

        # --- 4. 生成重建網格 ---
        # 我們在 PCA 的 XY 平面上生成網格來重建曲面
        # grid_size 取決於該組件的投影大小，這裡動態計算
        min_x, max_x = x_pca.min(), x_pca.max()
        min_y, max_y = y_pca.min(), y_pca.max()

        # 密度設為 1.0 (每個像素採樣一次)
        step = 0.8  # 稍微密一點可以填補空隙
        grid_x_range = torch.arange(min_x, max_x + step, step, device=device, dtype=torch.float64)
        grid_y_range = torch.arange(min_y, max_y + step, step, device=device, dtype=torch.float64)

        grid_x, grid_y = torch.meshgrid(grid_x_range, grid_y_range, indexing='ij')
        grid_x_flat = grid_x.flatten()
        grid_y_flat = grid_y.flatten()

        # 計算擬合後的 Z 值
        A_grid_list = []
        for i in range(degree + 1):
            for j in range(degree + 1 - i):
                term = ((grid_x_flat / x_scale) ** i) * ((grid_y_flat / y_scale) ** j)
                A_grid_list.append(term)

        # 預測 Z
        pred_z_flat = (torch.stack(A_grid_list, dim=1) @ coeffs)

        # --- 5. 轉回原始空間 ---
        # 組合: [x', y', z_pred]
        reconstructed_pca = torch.stack([grid_x_flat, grid_y_flat, pred_z_flat], dim=-1)

        # 逆旋轉 + 加回均值
        reconstructed_global = (reconstructed_pca @ V.T) + mean

        return reconstructed_global.cpu().numpy()

    except Exception as e:
        print(f"擬合失敗，跳過此組件: {e}")
        return None


def fill_hole_8slice(vol):
    """使用 8-slice 邏輯分段執行 binary_fill_holes。"""
    vol_bool = vol > 0
    output = np.zeros_like(vol_bool)

    slices = _octant_slices(vol_bool.shape, (2, 2, 2))
    struct_26 = ndimage.generate_binary_structure(3, 3)

    for slc in slices:
        block = vol_bool[slc]
        if np.any(block):
            output[slc] = binary_fill_holes(block, structure=struct_26)

    return output.astype(np.uint8)


import numpy as np


def clear_boundary_faces(vol: np.ndarray, margin: int = 3) -> np.ndarray:
    """
    將 3D Volume 六個面的邊界區域 (距離邊緣 <= margin) 設為 0。

    Args:
        vol (np.ndarray): 輸入的 3D 陣列 (Z, Y, X)。
        margin (int): 邊界寬度，預設為 2 (即 <= 2vx)。
        inplace (bool): 是否直接修改原陣列以節省記憶體。預設 False。

    Returns:
        np.ndarray: 處理後的陣列。
    """
    vol = vol.copy()
    vol = vol > 0
    # 確保 vol 是 3D
    if vol.ndim != 3:
        raise ValueError(f"Input volume must be 3D, but got shape {vol.shape}")

    # 1. Z 軸邊界 (上下)
    vol[:margin, :, :] = False
    vol[-margin:, :, :] = False

    # 2. Y 軸邊界 (前後)
    vol[:, :margin, :] = False
    vol[:, -margin:, :] = False

    # 3. X 軸邊界 (左右)
    vol[:, :, :margin] = False
    vol[:, :, -margin:] = False
    return vol.astype(np.uint8)


def fill_hole_8slice_keep_one_per_slice(vol):
    """
        1. 複製 vol 並移除小於 min_size (5000) 的物件 -> clean_vol
        2. 在 clean_vol 上執行 8-slice 分塊邏輯：
           - 找出所有洞
           - 計算洞的體積
           - 決定保留「體積最小」的那個洞 (不補)
           - 產生「被填補的洞」的遮罩 (fill_mask)
        3. 將 fill_mask 與 原始 vol 做 OR 運算回傳
        """
    # 0. 基礎設定
    vol_bool = vol > 0
    struct_26 = ndimage.generate_binary_structure(3, 3)

    # 建立一個全域的遮罩，用來存放「決定要補起來的洞」
    total_fill_mask = np.zeros_like(vol_bool)

    # --- 第一階段：前處理 (移除小物件) ---
    # 這裡確保雜訊不影響洞的判斷
    print("正在移除小物件以進行判斷...")
    clean_vol = remove_small_objects(vol_bool, min_size=5000, connectivity=1)

    # 取得切片
    slices = _octant_slices(clean_vol.shape, (2, 2, 2))

    # --- 第二階段：在乾淨的圖上計算「要補哪些洞」 ---
    for slc in slices:
        # 注意：這裡我們只看 clean_vol
        block = clean_vol[slc]

        # 如果區塊全空，跳過
        if not np.any(block):
            continue

        # 1. 全補
        filled_block = ndimage.binary_fill_holes(block, structure=struct_26)

        # 2. 找出洞 (填補後 - 原圖)
        holes_mask = filled_block & ~block

        # 如果沒有洞，跳過
        if not np.any(holes_mask):
            continue

        # 3. 標記所有的洞 (不再需要標記宿主物件)
        labeled_holes, num_holes = ndimage.label(holes_mask, structure=struct_26)

        # 預設：所有洞都要補 (稍後把要保留的那個洞挖掉)
        holes_to_fill_in_this_slice = holes_mask.copy()

        # 4. 邏輯判斷：保留體積最小的洞
        if num_holes > 0:
            # 計算每個洞的體積
            # sizes[0] 是背景，sizes[1:] 是各個洞的體積
            sizes = np.bincount(labeled_holes.ravel())

            # 找出最小洞的 Label
            # argmin 回傳的是索引，因為我們切片了 [1:]，所以索引要 +1 才是 Label
            target_hole_label = np.argmin(sizes[1:]) + 1

            # 從「要補的洞」清單中，移除這個「要保留的洞」
            # 將該 Label 的位置設為 False (不補)
            holes_to_fill_in_this_slice[labeled_holes == target_hole_label] = False

        # 5. 將決定好要補的洞，寫入全域遮罩
        total_fill_mask[slc] = holes_to_fill_in_this_slice

    # --- 第三階段：合併回原圖 ---
    # 結果 = 原始圖 OR 填補遮罩 (這樣原本 < 5000 的物件也會回來，且被選中的洞也被補上了)
    final_result = vol_bool | total_fill_mask

    return final_result.astype(np.uint8)


# # 1. 將 bm 宣告為全域變數，但在各進程內延遲載入
# _worker_bm = None
#
#
# def _get_bm_model():
#     """確保每個子進程都有自己的 bm 實體"""
#     global _worker_bm
#     if _worker_bm is None:
#         _worker_bm = load_betti_matching()
#     return _worker_bm
#
#
# # ==========================================
# # 2. 高斯修補副程式 (維持回傳邏輯)
# # ==========================================
# def _apply_gaussian_patch_and_return(chunk, center_coord, sigma_val):
#     z, y, x = center_coord
#     d, h, w = chunk.shape
#     r = 2
#
#     z_min, z_max = max(0, z - r), min(d, z + r + 1)
#     y_min, y_max = max(0, y - r), min(h, y + r + 1)
#     x_min, x_max = max(0, x - r), min(w, x + r + 1)
#
#     local_patch = chunk[z_min:z_max, y_min:y_max, x_min:x_max].astype(np.float32)
#     smoothed_patch = gaussian_filter(local_patch, sigma=sigma_val)
#
#     # 修改局部
#     chunk[z_min:z_max, y_min:y_max, x_min:x_max] |= (smoothed_patch > 0.5)
#     return chunk
#
#
# # ==========================================
# # 3. Worker 任務 (不再接收 bm_model)
# # ==========================================
# def _process_chunk_worker(args):
#     """
#     args 現在只包含 (slc_idx, chunk_data, sigma)
#     """
#     slc_idx, chunk, sigma = args
#
#     # 獲取該進程專用的 bm 物件
#     bm = _get_bm_model()
#
#     # 偵測破洞 (依據你提供的邏輯：自己跟自己 match 找特徵)
#     topo_pr = (~chunk).astype(np.uint8)
#     result = bm.compute_matching(topo_pr, topo_pr)
#
#     # 取得 Matched 座標 (依據你最新的程式碼需求)
#     if len(result.input1_matched_birth_coordinates) > 1:
#         local_coords = result.input1_matched_birth_coordinates[1]
#         for l_coord in local_coords:
#             chunk = _apply_gaussian_patch_and_return(chunk, l_coord, sigma)
#     print("處理完成")
#     return slc_idx, chunk
#
#
# # ==========================================
# # 4. 主函數
# # ==========================================
# def fill_holes_multiprocess_return(vol, split_parts=(2, 2, 2), sigma=1.0, n_procs=8):
#     """
#     注意：參數中移除了 bm，改由子進程自行載入
#     """
#     output_vol = np.zeros_like(vol)
#     slices = _octant_slices(vol.shape, split_parts)
#
#     # 準備參數包 (剔除 bm_model)
#     task_args = [
#         (i, vol[slc].copy(), sigma)
#         for i, slc in enumerate(slices)
#     ]
#
#     print(f"[-] 開始多進程修補 (進程數: {n_procs})...")
#
#     # 使用 'spawn' 或 'forkserver' 模式在某些系統更穩定，但 Linux 預設 fork 即可
#     with multiprocessing.Pool(processes=n_procs) as pool:
#         results = pool.map(_process_chunk_worker, task_args)
#
#     print("[-] 正在組裝最終體積...")
#     for idx, processed_chunk in results:
#         slc = slices[idx]
#         output_vol[slc] = processed_chunk
#
#     print("[-] 處理完成。")
#     return output_vol


def process_gaussian_octants_dual_threshold(
        vol: np.ndarray,
        sigma: float = 1.0,
        inner_threshold: float = 0.5,
        border_threshold: float = 0.1,
        border: int = 8
) -> np.ndarray:
    """
    將 3D 體積切塊處理：
    1. 內部區域 (Inner): 使用 inner_threshold (預設 0.5)
    2. 邊界區域 (Border): 使用 border_threshold (預設 0.1)
    """
    output_vol = vol.copy()

    # 取得 8 塊區域的切片範圍
    slices_list = _octant_slices(vol.shape, splits=(2, 2, 2))

    for sl in slices_list:
        # 取出該區塊
        sub_vol = vol[sl]
        depth, height, width = sub_vol.shape

        # 防呆：確保區塊大於兩倍 border
        if height <= 2 * border or width <= 2 * border or depth <= 2 * border:
            # 若區塊太小無法區分 border，則統一使用 border_threshold 處理或跳過
            blurred_small = ndi.gaussian_filter(sub_vol.astype(float), sigma=sigma)
            output_vol[sl] = (blurred_small > border_threshold).astype(vol.dtype)
            continue

        # 核心：高斯模糊運算
        blurred_sub = ndi.gaussian_filter(sub_vol.astype(float), sigma=sigma)

        # 建立該區塊的結果容器，先統一用 border_threshold 處理整塊
        processed_block = (blurred_sub > border_threshold).astype(vol.dtype)

        # 定義中心區域 (Inner area) 的切片
        inner_z = slice(border, -border if border > 0 else None)
        inner_h = slice(border, -border if border > 0 else None)
        inner_w = slice(border, -border if border > 0 else None)
        inner_slice = (inner_z, inner_h, inner_w)

        # 針對「中心區域」覆蓋使用 inner_threshold 的結果
        # 這會讓中心區域與邊界區域有不同的二值化敏感度
        processed_block[inner_slice] = (blurred_sub[inner_slice] > inner_threshold).astype(vol.dtype)

        # 將處理完的塊填回大圖
        output_vol[sl] = processed_block

    return output_vol


def fill_2x2diag(
        mask_3d: np.ndarray,
        axis: int = 0,
        gap: int = 0,
) -> np.ndarray:
    """
    掃描相鄰切片，修補 2x2 對角跳空 (diagonal gap) 問題。

    Args:
        mask_3d: 3D binary mask，shape = (z, y, x)
        axis:    沿哪個軸做切片比較。0 = z 軸（預設），1 = y 軸，2 = x 軸
        gap:     prev 與 curr 之間允許的中間層數。
                 0 = 原始行為（相鄰兩層）。
                 N = prev 與 curr 相距 N+1 層，中間 N 層需滿足
                     連續相鄰對在該 2x2 的 AND 總和 > 1，
                     確認有連續性後再判斷 prev/curr 的 3/3/4 對角條件，
                     全部通過才將整段全設為 True。

    Returns:
        修補後的 3D binary mask（不修改原始輸入）
    """
    if axis not in (0, 1, 2):
        raise ValueError(f"axis 必須是 0、1 或 2，收到 {axis}")
    if gap < 0:
        raise ValueError(f"gap 必須 >= 0，收到 {gap}")

    work = np.moveaxis(mask_3d, axis, 0).copy().astype(bool)
    d, h, w = work.shape

    step = gap + 1  # prev 與 curr 的 index 距離
    corner_offsets = [(0, 0), (0, 1), (1, 0), (1, 1)]

    for z in range(step, d):
        prev_idx = z - step
        curr_idx = z  # inclusive，共 step+1 層需要填補

        prev = work[prev_idx].astype(np.uint8)
        curr = work[curr_idx].astype(np.uint8)
        orr = prev | curr

        for dr, dc in corner_offsets:
            r0, r1 = dr, h - 1 - dr  # 2x2 左上角的 row 範圍 [r0, r1)
            c0, c1 = dc, w - 1 - dc

            if r1 <= r0 or c1 <= c0:
                continue

            def box_sum(arr: np.ndarray) -> np.ndarray:
                return (arr[r0: r1, c0: c1]
                        + arr[r0: r1, c0 + 1: c1 + 1]
                        + arr[r0 + 1: r1 + 1, c0: c1]
                        + arr[r0 + 1: r1 + 1, c0 + 1: c1 + 1])

            # ── 條件一：prev / curr 滿足 3/3/4 對角規則 ──────────────────
            cond = (box_sum(prev) == 3) & (box_sum(curr) == 3) & (box_sum(orr) == 4)

            if not np.any(cond):
                continue

            # ── 條件二：中間所有相鄰對的 AND 在該 2x2 內 > 1 ─────────────
            # 遍歷 (prev_idx, prev_idx+1), ..., (curr_idx-1, curr_idx)
            # gap=0 時此迴圈不執行
            if gap > 0:
                for i in range(prev_idx, curr_idx):
                    a = work[i].astype(np.uint8)
                    b = work[i + 1].astype(np.uint8)
                    cond = cond & (box_sum(a & b) > 1)

                    if not np.any(cond):
                        break  # 提早結束，此 corner 已無命中點

            # ── 填補：整段 [prev_idx, curr_idx] 的該 2x2 全設為 True ─────
            hit_rows, hit_cols = np.where(cond)
            if len(hit_rows) == 0:
                continue

            abs_rows = hit_rows + r0
            abs_cols = hit_cols + c0

            for rr, cc in zip(abs_rows, abs_cols):
                work[prev_idx: curr_idx + 1, rr: rr + 2, cc: cc + 2] = True

    return np.moveaxis(work, 0, axis)


def apply_post_processing(
        pred: np.ndarray,
        min_size: int,
        split_paper: bool,
        split_paper_iter: int = 1,
        split_high_prob: float = 0.8,
        prob: np.ndarray = None,  # 機率圖
        fg_threshold: float = None,  # 門檻值 (必填以啟用取代功能)
        skip: bool = False,
        line_norm: bool = False,
        y_axis_closing: bool = False,
        y_axis_fill_holes: bool = False,
        z_axis_fill_holes: bool = False,
        fill_hole: bool = False,
        line_endpoints_repair: bool = False,
        line_mask_repair: bool = False,
        repair_settings: dict = None,
        sandwich: bool = False,
        sandwich_iterations: int = 1,
        gaussian: bool = False,
        clear_boundary: bool = False,
        dig2x2: bool = False
) -> np.ndarray:
    """
    整合後處理流程。
    """
    # 複製並初始化
    vol = pred.copy()
    vol[vol == 2] = 0  # 清除 Class 2 (若有)

    if skip:
        return vol


    if min_size > 0:
        vol_bool = vol > 0
        vol_bool = remove_small_objects(vol_bool, min_size=5000, connectivity=1)

        vol = vol_bool.astype(np.uint8)
    else:
        vol = (vol > 0).astype(np.uint8)

    # ==========================================
    # 3. 斷線修復
    # ==========================================
    if line_endpoints_repair:
        vol = execute_line_repair(vol, settings=repair_settings)
        # vol = execute_line_repair_8slice(vol, settings=repair_settings)

        vol = (vol > 0).astype(np.uint8)

    if line_mask_repair:
        vol = line_repair_msk(vol, min_area=10, max_link_dist=120.0, pass_iters=2, line_thickness=2)
        vol = (vol > 0).astype(np.uint8)

    # ==========================================
    # 4. 形態學與幾何優化
    # ==========================================
    if line_norm:
        Z_15, _ = normalize_segments_3d(vol, radius=1.5, border=8, axis=0, sigma=0, repair=True, repair_dijkstra=False,
                                        max_dist=30, max_angle_deg=45, safe_distance=2, iterations=1)

        Y_15, _ = normalize_segments_3d(vol, radius=1.5, border=8, axis=1, sigma=0, repair=True, repair_dijkstra=False,
                                        max_dist=30, max_angle_deg=45, safe_distance=2, iterations=1)

        vol = vol | Z_15 | Y_15
        vol = vol.astype(np.uint8)

    if sandwich:
        # 建議 gap=2 以應對連續斷層
        vol = robust_mask_refine(vol, max_gap=3, iterations=sandwich_iterations, min_neighbors=9, kernel_size=5)

        vol = vol.astype(np.uint8)

    if gaussian:
        vol = process_gaussian_octants_dual_threshold(
            vol,
            sigma=1,
            inner_threshold=0.5,
            border_threshold=0.5,
            border=8)

    if split_paper:  # 註：這裡依照您提供的 code 原樣保留變數名
        foreground_prob = prob[1]
        high_conf_mask = (foreground_prob > split_high_prob)

        # 建議這裡也考慮使用 8-slice 版本的 remove_small_objects
        high_conf_mask = remove_small_objects(high_conf_mask, min_size=5000, connectivity=1)
        high_conf_labels = measure_label(high_conf_mask).astype(np.int32)

        # --- 改用 8-slice 版本 ---
        vol, debug_paths = get_split_paper_8slice(
            vol,
            high_conf_labels,
            ray_len=64,
            max_iter=split_paper_iter,
            cleanup_iter=2
        )
        # -----------------------

        vol = vol > 0
        vol = vol.astype(np.uint8)

    if y_axis_closing:
        vol = apply_y_axis_closing(vol, iterations=1)
        vol = vol.astype(np.uint8)

    if y_axis_fill_holes:
        vol = apply_y_axis_fill_holes(vol)
        vol = vol.astype(np.uint8)

    if z_axis_fill_holes:
        vol = apply_z_axis_fill_holes(vol)
        vol = vol.astype(np.uint8)

    if fill_hole:
        vol = fill_hole_8slice_keep_one_per_slice(vol)
        # vol = binary_fill_holes(vol)
        # vol = vol.astype(np.uint8)

    if min_size > 0:
        vol_bool = vol > 0
        vol_bool = remove_small_objects(vol_bool, min_size=min_size, connectivity=1)
        # vol_bool = clear_boundary_faces(vol_bool,margin=3)
        vol_bool = remove_small_8slice(vol_bool, min_size=10)

        vol = vol_bool.astype(np.uint8)
    else:
        vol = (vol > 0).astype(np.uint8)
    if dig2x2:
        vol = fill_2x2diag(vol, axis=0, gap=0)
        vol = fill_2x2diag(vol, axis=0, gap=1)
        vol = fill_2x2diag(vol, axis=1, gap=0)
        vol = fill_2x2diag(vol, axis=1, gap=1)
        vol.astype(np.uint8)

    return vol.astype(np.uint8)

In [6]:
from pathlib import Path
import tifffile as tiff
from skimage import measure
import zipfile
import csv

# ==========================================
# 2. 主程式設定與執行 (Main Execution)
# ==========================================

# --- 設定路徑 ---
source_dir = './nnUNet_upload/3d_fullres_TTA'
processed_dir = './processed_labels'
output_zip = './submission.zip'
stats_file = './processing_stats.csv'

os.makedirs(processed_dir, exist_ok=True)

# --- 設定後處理參數 ---
MIN_SIZE = 5000         # 移除小於此體積 (voxel數) 的雜訊


# 準備統計數據列表
stats_data = []

print(f"{'Filename':<30} | {'Before':<10} | {'After':<10} | {'Diff':<10}")
print("-" * 70)

tif_files = list(Path(source_dir).glob('*.tif'))

for tif_path in tif_files:
    # A. 讀取數據
    data = tiff.imread(str(tif_path))

    # 確保是二值化 (0 或 1)
    binary_mask = (data > 0).astype(np.uint8)

    # --- 統計處理前物件數量 ---
    # label() 用於計算連通域數量
    _, count_before = measure.label(binary_mask, return_num=True)

    # B. 執行新的後處理函數
    processed_data = apply_post_processing(
        binary_mask,
        skip=False,
        min_size=MIN_SIZE,
        split_paper=False,
        split_paper_iter=1,
        split_high_prob=0.9,
        line_norm=True,
        line_mask_repair=False,
        line_endpoints_repair=False,
        y_axis_closing=False,
        z_axis_fill_holes=False,
        y_axis_fill_holes=False,
        sandwich=True,
        sandwich_iterations=5,
        fill_hole=False,
        gaussian=True,
        dig2x2=True
    )

    # --- 統計處理後物件數量 ---
    _, count_after = measure.label(processed_data, return_num=True)

    # 記錄統計數據
    diff = count_after - count_before
    stats_data.append([tif_path.name, count_before, count_after, diff])

    # C. 儲存處理後的圖檔 (只有 0 與 1)
    save_path = os.path.join(processed_dir, tif_path.name)
    tiff.imwrite(save_path, processed_data, compression='zlib')

    print(f"{tif_path.name:<30} | {count_before:<10} | {count_after:<10} | {diff:<+10}")

# ==========================================
# 3. 匯出結果 (ZIP & CSV)
# ==========================================

# 建立 ZIP 檔案
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tif_file in Path(processed_dir).glob('*.tif'):
        zf.write(tif_file, arcname=tif_file.name)

# 輸出 CSV 統計報表
with open(stats_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Filename', 'Count_Before', 'Count_After', 'Difference'])
    writer.writerows(stats_data)

print("\n" + "="*70)
print(f"處理完成！使用新演算法")
print(f"總處理檔案數: {len(tif_files)}")
print(f"ZIP 檔案路徑: {output_zip}")
print(f"CSV 統計報表: {stats_file}")
print("="*70)

Filename                       | Before     | After      | Diff      
----------------------------------------------------------------------
1407735.tif                    | 62         | 7          | -55       

處理完成！使用新演算法
總處理檔案數: 1
ZIP 檔案路徑: ./submission.zip
CSV 統計報表: ./processing_stats.csv
